<div align="center">

# Laboratorio 9 · Embeddings

**Curso:** Natural Language Processing  
**Estudiante:** Jose Tánchez  
**Carné:** 20230005

</div>



# 1. Preparación de los corpus

Prepararemos las noticias y *Don Quijote de la Mancha* como **listas de oraciones**, cada una representada por una lista de tokens en su orden original. Las ventanas de contexto respetarán los límites de cada oración.

Las salidas muestran la preparación de ambos corpus, la cobertura de los vocabularios y la construcción de pares skip-gram. El análisis de la sección 1.8 se basa en los resultados de esta ejecución.

## 1.1 Configuración y archivos

Se reutilizan `Lab1/df_total.csv` y `Lab5/don-quijote.txt`. El notebook puede ejecutarse desde la raíz del repositorio o desde `Lab9`.

Se requieren **pandas** y **NLTK**. La comprobación de SUC requiere además **gensim** y el archivo `embeddings-m-model.vec`. Si faltan paquetes, se pueden instalar desde una celda con `%pip install pandas nltk gensim`. El modelo preentrenado debe descargarse desde el enlace indicado en la guía y no debe subirse al repositorio.

### Modelo SUC utilizado

Se utilizó el [modelo fastText SUC M en formato `.vec`, de 100 dimensiones](https://zenodo.org/record/3234051/files/embeddings-m-model.vec?download=1), guardado localmente como `Lab9/embeddings-m-model.vec`. El archivo completo pesa aproximadamente **1.1 GB**; para esta práctica se cargaron **200,000 entradas**.

El modelo preentrenado y sus variantes de formato o archivos comprimidos se excluyen del repositorio mediante `.gitignore`. La documentación `emb-from-suc.md` contiene los enlaces de descarga y permite identificar el recurso. Los vectores propios de SVD, `ppmi_svd_100.kv`, sí forman parte de los resultados del laboratorio.


In [48]:
from pathlib import Path
from collections import Counter
import os
import re
import unicodedata

import nltk
import pandas as pd
from IPython.display import display

candidatas = [Path.cwd(), *Path.cwd().parents]
RAIZ = next(
    (ruta for ruta in candidatas
     if (ruta / "Lab1/df_total.csv").is_file()
     and (ruta / "Lab5/don-quijote.txt").is_file()),
    None,
)
if RAIZ is None:
    raise FileNotFoundError("Abre el notebook desde el repositorio NLP o desde Lab9.")

RUTA_NOTICIAS = RAIZ / "Lab1/df_total.csv"
RUTA_QUIJOTE = RAIZ / "Lab5/don-quijote.txt"
# Cambiar esta ruta si el modelo se encuentra en otro directorio.
RUTA_SUC = Path(os.environ.get(
    "SUC_PATH", str(RAIZ / "Lab9/embeddings-m-model.vec")
)).expanduser()
LIMITE_SUC = 200_000  # Usar 100_000 si hay poca memoria.

print(f"Raíz del proyecto: {RAIZ}")
print(f"Noticias: {RUTA_NOTICIAS.relative_to(RAIZ)}")
print(f"Quijote: {RUTA_QUIJOTE.relative_to(RAIZ)}")

Raíz del proyecto: /Users/josetanchezz/Desktop/NLP
Noticias: Lab1/df_total.csv
Quijote: Lab5/don-quijote.txt


## 1.2 Carga y revisión de los documentos

Cada noticia con texto es un documento; el libro completo es un documento. Se excluyen las noticias vacías y se reporta cuántas fueron descartadas. Se mantienen los textos repetidos para conservar el corpus usado en el Laboratorio 1, pero se informa su cantidad porque pueden aumentar la frecuencia de algunos contextos.

En el Quijote se retiran la cabecera y la licencia de Project Gutenberg. Se conservan el título, los preliminares y el texto literario en español.

In [49]:
df_original = pd.read_csv(RUTA_NOTICIAS)
if "news" not in df_original.columns:
    raise ValueError(f"No se encontró la columna news: {df_original.columns.tolist()}")

textos = df_original["news"].fillna("").astype(str).str.strip()
mascara_validos = textos.ne("")
noticias = df_original.loc[mascara_validos].copy()
noticias["news"] = textos.loc[mascara_validos]
noticias = noticias.reset_index(drop=True)
documentos_noticias = noticias["news"].tolist()

texto_quijote_original = RUTA_QUIJOTE.read_text(encoding="utf-8-sig")
inicio = texto_quijote_original.find("El ingenioso hidalgo don Quijote de la Mancha")
fin = re.search(
    r"(?:End of Project Gutenberg[^\n]*|\*\*\*\s*END OF (?:THIS|THE) PROJECT GUTENBERG)",
    texto_quijote_original,
)
if inicio == -1 or fin is None or fin.start() <= inicio:
    raise ValueError("Revisar los delimitadores de la edición del Quijote antes de continuar.")
texto_quijote = texto_quijote_original[inicio:fin.start()].strip()
documentos_quijote = [texto_quijote]

revision_carga = pd.DataFrame([
    {"Corpus": "Noticias", "Documentos originales": len(df_original),
     "Documentos utilizados": len(noticias),
     "Documentos vacíos excluidos": int((~mascara_validos).sum()),
     "Textos duplicados adicionales": int(noticias["news"].duplicated().sum())},
    {"Corpus": "Quijote", "Documentos originales": 1,
     "Documentos utilizados": 1, "Documentos vacíos excluidos": 0,
     "Textos duplicados adicionales": 0},
])
display(revision_carga)
print("Inicio del Quijote utilizado:\n", texto_quijote[:400])
print("\nFinal del Quijote utilizado:\n", texto_quijote[-300:])

,Corpus,Documentos originales,Documentos utilizados,Documentos vacíos excluidos,Textos duplicados adicionales
0,Noticias,1217,1213,4,76
1,Quijote,1,1,0,0


Inicio del Quijote utilizado:
 El ingenioso hidalgo don Quijote de la Mancha


TASA

Yo, Juan Gallo de Andrada, escribano de Cámara del Rey nuestro señor, de
los que residen en su Consejo, certifico y doy fe que, habiendo visto por
los señores dél un libro intitulado El ingenioso hidalgo de la Mancha,
compuesto por Miguel de Cervantes Saavedra, tasaron cada pliego del dicho
libro a tres maravedís y medio; el cual tiene ochenta 

Final del Quijote utilizado:
 o de
sus escritos enteramente, como deseaba, pues no ha sido otro mi deseo que
poner en aborrecimiento de los hombres las fingidas y disparatadas
historias de los libros de caballerías, que, por las de mi verdadero don
Quijote, van ya tropezando, y han de caer del todo, sin duda alguna. Vale.



Fin


## 1.3 Decisiones de normalización

Retomamos la tokenización, las minúsculas y la eliminación de puntuación del Laboratorio 1, pero cambiamos las decisiones que afectarían el aprendizaje de contextos.

| Elemento | Decisión | Justificación |
|---|---|---|
| Oraciones | Segmentar antes de eliminar puntuación | Evita crear pares entre el final de una oración y el inicio de otra. |
| Mayúsculas | Convertir a minúsculas | Agrupa variantes y reduce la dispersión en corpus pequeños. En SUC se encontró `españa`, pero no `España` ni `ESPAÑA`, dentro de las 200,000 entradas cargadas. |
| Puntuación | Retirar después de segmentar | Las ventanas se construirán sobre palabras y números. |
| Números | Conservar secuencias de dígitos, sin un token artificial | Mantiene referencias numéricas y evita introducir un marcador sin vector en SUC. Los separadores decimales se eliminan y separan los grupos numéricos. |
| Tildes, diéresis y ñ | Conservar y normalizar Unicode a NFC | Evita confundir palabras como `sí` y `si` o `año` y `ano`. |
| Stopwords | Conservar | Quitarlas acerca palabras que no eran vecinas y elimina información sintáctica. El submuestreo de Word2Vec podrá reducir su peso durante el entrenamiento. |
| Lematización | No aplicar | Conserva número, género y tiempos verbales, necesarios para estudiar relaciones morfológicas y analogías. |

`gobiernos` y `gobernó` permanecen distintos: el primero es un sustantivo plural y el segundo, un verbo en pasado. Una lematización correcta daría `gobierno` y `gobernar`, respectivamente; agruparlos en una sola forma borraría también la diferencia de categoría gramatical.

**Compatibilidad con SUC.** La comprobación de la sección 1.7 respalda el uso de minúsculas para las variantes examinadas. Las formas con y sin tilde aparecen como claves de vocabulario, por lo que no se eliminan las tildes ni se fusionan esas grafías. Las comparaciones usarán exactamente los mismos tokens en todos los modelos y reportarán las palabras fuera del vocabulario.

In [50]:
# Punkt segmenta en español sin necesitar un modelo de spaCy.
try:
    nltk.sent_tokenize("Primera oración. Segunda oración.", language="spanish")
except LookupError:
    nltk.download("punkt", quiet=True, raise_on_error=True)
    nltk.download("punkt_tab", quiet=True, raise_on_error=True)

PATRON_TOKEN = re.compile(r"[a-záéíóúüñ0-9]+", flags=re.IGNORECASE)


def preparar_documento(texto):
    """Segmentar primero y normalizar después, conservando el orden."""
    texto = unicodedata.normalize("NFC", texto)
    # En el CSV hay finales concatenados, como «imaginamos.Para».
    # Esta reparación es heurística; puede afectar abreviaturas sin espacio.
    texto = re.sub(r"(?<=[.!?])(?=[A-ZÁÉÍÓÚÜÑ¿¡])", " ", texto)
    resultado = []
    for oracion in nltk.sent_tokenize(texto, language="spanish"):
        tokens = PATRON_TOKEN.findall(oracion.lower())
        if tokens:
            resultado.append(tokens)
    return resultado


# Se conserva la estructura por documento para usos posteriores.
noticias["oraciones_tokens"] = [
    preparar_documento(texto) for texto in documentos_noticias
]
quijote_por_documento = [preparar_documento(texto) for texto in documentos_quijote]

# Estas dos listas de listas son las entradas para gensim.
oraciones_noticias = [
    oracion for documento in noticias["oraciones_tokens"] for oracion in documento
]
oraciones_quijote = [
    oracion for documento in quijote_por_documento for oracion in documento
]
corpus_oraciones = {"Noticias": oraciones_noticias, "Quijote": oraciones_quijote}

print("Formato de entrada: list[list[str]]")
for nombre, oraciones in corpus_oraciones.items():
    print(f"\n{nombre}: primeras dos oraciones tokenizadas")
    for oracion in oraciones[:2]:
        print(oracion)

Formato de entrada: list[list[str]]

Noticias: primeras dos oraciones tokenizadas
['durante', 'el', 'foro', 'la', 'banca', 'articulador', 'empresarial', 'para', 'el', 'desarrollo', 'sostenible', 'el', 'director', 'de', 'sostenibilidad', 'y', 'clientes', 'globales', 'de', 'bbva', 'en', 'colombia', 'andrés', 'garcía', 'aseguró', 'que', 'es', 'importante', 'entender', 'que', 'la', 'sostenibilidad', 'no', 'la', 'podemos', 'asociar', 'a', 'mayores', 'costos']
['yo', 'creo', 'que', 'el', 'no', 'tener', 'un', 'concepto', 'de', 'negocio', 'sostenible', 'puede', 'tener', 'un', 'mayor', 'impacto', 'de', 'lo', 'que', 'imaginamos']

Quijote: primeras dos oraciones tokenizadas
['el', 'ingenioso', 'hidalgo', 'don', 'quijote', 'de', 'la', 'mancha', 'tasa', 'yo', 'juan', 'gallo', 'de', 'andrada', 'escribano', 'de', 'cámara', 'del', 'rey', 'nuestro', 'señor', 'de', 'los', 'que', 'residen', 'en', 'su', 'consejo', 'certifico', 'y', 'doy', 'fe', 'que', 'habiendo', 'visto', 'por', 'los', 'señores', 'dél', 

La segmentación de oraciones es automática y puede equivocarse con abreviaturas, encabezados o puntuación irregular. En el libro los saltos de línea de la edición no se toman por sí solos como finales de oración.

## 1.4 Tamaño de los corpus

Un **token** es una aparición de una palabra o número; un **tipo** es una forma distinta después de normalizar. Se cuentan solamente las oraciones que conservan al menos un token.

In [51]:
frecuencias_corpus = {
    nombre: Counter(token for oracion in oraciones for token in oracion)
    for nombre, oraciones in corpus_oraciones.items()
}
documentos_por_corpus = {"Noticias": len(documentos_noticias), "Quijote": 1}
resumen_corpus = pd.DataFrame([
    {"Corpus": nombre, "Documentos": documentos_por_corpus[nombre],
     "Oraciones": len(oraciones),
     "Tokens": sum(frecuencias_corpus[nombre].values()),
     "Tipos": len(frecuencias_corpus[nombre])}
    for nombre, oraciones in corpus_oraciones.items()
])
display(resumen_corpus)

,Corpus,Documentos,Oraciones,Tokens,Tipos
0,Noticias,1213,21855,637610,29507
1,Quijote,1,9550,381230,22943


## 1.5 Vocabulario y frecuencia mínima

Se construye un vocabulario independiente para cada corpus y para cada valor de `min_count`. El porcentaje excluido se calcula sobre **apariciones de tokens**, no sobre tipos:

$$\%\text{ de tokens fuera}=100\frac{\sum_{w:f(w)<m} f(w)}{\sum_w f(w)}.$$

En esta etapa se mide la cobertura; las listas originales no se filtran ni se reordenan. Gensim aplicará el umbral al construir su vocabulario durante el entrenamiento.

In [52]:
MIN_COUNTS = (1, 2, 5)
vocabularios = {}
filas_vocabulario = []
for nombre, frecuencias in frecuencias_corpus.items():
    total = sum(frecuencias.values())
    vocabularios[nombre] = {}
    for minimo in MIN_COUNTS:
        vocabulario = {palabra for palabra, frecuencia in frecuencias.items()
                       if frecuencia >= minimo}
        vocabularios[nombre][minimo] = vocabulario
        fuera = sum(frecuencia for palabra, frecuencia in frecuencias.items()
                    if palabra not in vocabulario)
        filas_vocabulario.append({
            "Corpus": nombre, "min_count": minimo,
            "Tamaño del vocabulario": len(vocabulario),
            "Tokens fuera": fuera,
            "Tokens fuera (%)": round(100 * fuera / total, 4) if total else 0.0,
        })
resumen_vocabularios = pd.DataFrame(filas_vocabulario)
display(resumen_vocabularios)

# Ejemplo conceptual tokens -> IDs. Gensim asignará sus propios IDs.
palabra_a_id = {palabra: i for i, (palabra, frecuencia) in enumerate(
    frecuencias_corpus["Noticias"].most_common()
)}
ejemplo_tokens = oraciones_noticias[0]
display(pd.DataFrame({"Token": ejemplo_tokens,
                      "ID ilustrativo": [palabra_a_id[t] for t in ejemplo_tokens]}))

,Corpus,min_count,Tamaño del vocabulario,Tokens fuera,Tokens fuera (%)
0,Noticias,1,29507,0,0.0000
1,Noticias,2,17129,12378,1.9413
2,Noticias,5,8890,34319,5.3824
3,Quijote,1,22943,0,0.0000
4,Quijote,2,11760,11183,2.9334
5,Quijote,5,5301,28121,7.3764


,Token,ID ilustrativo
0,durante,61
1,el,3
2,foro,1582
3,la,1
4,banca,206
5,articulador,12811
6,empresarial,667
7,para,11
8,el,3
9,desarrollo,80


## 1.6 Pares skip-gram con código propio

Cada posición central se combina con todas las posiciones de contexto a una distancia máxima de dos tokens. Los pares son **dirigidos** y se mantienen sus repeticiones: dos posiciones distintas pueden producir el mismo par de palabras. Nunca se atraviesa el límite de una oración.

Se elige una oración de entre 8 y 15 tokens para poder mostrar **todos** sus pares sin truncar la tabla.

In [53]:
def pares_skipgram(tokens, ventana=2):
    """Producir pares (central, contexto), sin incluir la propia posición."""
    if ventana < 1:
        raise ValueError("La ventana debe ser positiva.")
    for i, central in enumerate(tokens):
        inicio = max(0, i - ventana)
        fin = min(len(tokens), i + ventana + 1)
        for j in range(inicio, fin):
            if i != j:
                yield central, tokens[j]


oracion_ejemplo = next(
    (oracion for oracion in oraciones_noticias if 8 <= len(oracion) <= 15),
    oraciones_noticias[0],
)
pares_ejemplo = list(pares_skipgram(oracion_ejemplo, ventana=2))
tabla_pares = pd.DataFrame(pares_ejemplo, columns=["Palabra central", "Contexto"])
tabla_pares.index = range(1, len(tabla_pares) + 1)
tabla_pares.index.name = "Par"
print("Oración normalizada:", " ".join(oracion_ejemplo))
print(f"Tokens: {len(oracion_ejemplo)} | Pares con ventana 2: {len(pares_ejemplo)}")
with pd.option_context("display.max_rows", None):
    display(tabla_pares)

Oración normalizada: la sostenibilidad nos abre oportunidades a mejores fuentes de financiamiento agregó
Tokens: 11 | Pares con ventana 2: 38


,Palabra central,Contexto
Par,,
1,la,sostenibilidad
2,la,nos
3,sostenibilidad,la
4,sostenibilidad,nos
5,sostenibilidad,abre
6,nos,la
7,nos,sostenibilidad
8,nos,abre
9,nos,oportunidades


### Cantidad de pares en todo el corpus de noticias

Para una oración de longitud $n$, cada distancia $d$ aporta $2(n-d)$ pares dirigidos. Con $m=\min(w,n-1)$:

$$N(n,w)=2\sum_{d=1}^{m}(n-d)=2mn-m(m+1).$$

Esta fórmula permite contar sin guardar millones de pares en memoria. Es el conteo de una pasada con **ventana fija**, sobre todos los tokens preparados, antes de `min_count` o submuestreo. El entrenamiento de gensim puede usar ventanas reducidas aleatoriamente, descartar tokens y repetir épocas, por lo que su número efectivo de ejemplos puede ser diferente.

In [54]:
def contar_pares(longitud, ventana):
    m = min(ventana, max(0, longitud - 1))
    return 2 * m * longitud - m * (m + 1)


# Comprobar la fórmula contra el generador usado en el ejemplo.
assert contar_pares(len(oracion_ejemplo), 2) == len(pares_ejemplo)
total_tokens_noticias = sum(map(len, oraciones_noticias))
comparacion_pares = pd.DataFrame([
    {"Ventana": ventana, "Tokens": total_tokens_noticias,
     "Pares de entrenamiento": sum(contar_pares(len(s), ventana)
                                    for s in oraciones_noticias)}
    for ventana in (2, 5)
])
comparacion_pares["Pares por token"] = (
    comparacion_pares["Pares de entrenamiento"] / total_tokens_noticias
).round(4)
display(comparacion_pares)

,Ventana,Tokens,Pares de entrenamiento,Pares por token
0,2,637610,2419436,3.7945
1,5,637610,5722750,8.9753


**¿Por qué Word2Vec no necesita datos etiquetados?** El propio orden del texto proporciona la señal de entrenamiento: cada palabra central permite obtener sus contextos observados. No se necesitan categorías anotadas por una persona. Con una ventana mayor cada posición puede originar más pares; cerca de los límites de oración hay menos vecinos disponibles. Este aprendizaje se denomina autosupervisado.

## 1.7 Comprobación del vocabulario fastText SUC

Se cargaron **200,000 entradas de SUC**, cada una con **100 dimensiones**. El modelo se carga una sola vez por configuración en la sesión, con un límite de vocabulario para controlar la memoria. La celda conserva una comprobación del archivo para facilitar la ejecución en otros equipos.

Se consulta `key_to_index` para comprobar la presencia **exacta** de cada grafía. La ausencia de una palabra en las primeras 200 000 entradas no demuestra que esté ausente del archivo completo. La cobertura se mide sin alterar los tokens de los corpus.

In [55]:
if not RUTA_SUC.is_file():
    print("PENDIENTE: no se encontró el archivo de fastText SUC.")
    print(f"Colócalo en {RUTA_SUC} o modifica RUTA_SUC y vuelve a ejecutar esta celda.")
else:
    from gensim.models import KeyedVectors

    firma_suc = (str(RUTA_SUC.resolve()), LIMITE_SUC)
    if globals().get("_firma_suc_cargada") != firma_suc:
        suc = KeyedVectors.load_word2vec_format(
            str(RUTA_SUC), limit=LIMITE_SUC
        )
        _firma_suc_cargada = firma_suc

    variantes = ["españa", "España", "espana", "ESPAÑA",
                 "tecnología", "tecnologia", "año", "ano"]
    revision_suc = pd.DataFrame([
        {"Grafía exacta": palabra, "Presente en SUC cargado": palabra in suc.key_to_index}
        for palabra in variantes
    ])
    print(f"Entradas cargadas: {len(suc):,} | Dimensiones: {suc.vector_size}")
    display(revision_suc)

    cobertura_suc = pd.DataFrame([
        {"Corpus": nombre,
         "Tipos cubiertos (%)": round(100 * sum(t in suc.key_to_index for t in frec)
                                      / len(frec), 2),
         "Tokens cubiertos (%)": round(100 * sum(n for t, n in frec.items()
                                                 if t in suc.key_to_index)
                                       / sum(frec.values()), 2)}
        for nombre, frec in frecuencias_corpus.items()
    ])
    display(cobertura_suc)
    print("Interpretar estas tablas antes de cerrar la justificación de compatibilidad con SUC.")

Entradas cargadas: 200,000 | Dimensiones: 100


,Grafía exacta,Presente en SUC cargado
0,españa,True
1,España,False
2,espana,True
3,ESPAÑA,False
4,tecnología,True
5,tecnologia,True
6,año,True
7,ano,True


,Corpus,Tipos cubiertos (%),Tokens cubiertos (%)
0,Noticias,86.1,98.70
1,Quijote,73.5,97.38


Interpretar estas tablas antes de cerrar la justificación de compatibilidad con SUC.


## 1.8 Análisis de los resultados

### Preparación y tamaño de los corpus

El archivo de noticias contenía **1,217 documentos**. Se excluyeron **4 textos vacíos**, por lo que se trabajó con **1,213 noticias**. También se encontraron **76 textos duplicados adicionales**, que se conservaron para mantener el corpus utilizado en el Laboratorio 1. Esto implica que algunos contextos repetidos tendrán más peso en el entrenamiento.

En el Quijote se utilizó el libro como **un solo documento**, retirando el material externo de Project Gutenberg y conservando los preliminares en español. Después de segmentar y normalizar se obtuvieron los siguientes tamaños:

| Corpus | Documentos utilizados | Oraciones | Tokens | Tipos |
|---|---:|---:|---:|---:|
| Noticias | 1,213 | 21,855 | 637,610 | 29,507 |
| Quijote | 1 | 9,550 | 381,230 | 22,943 |

Las noticias aportan más tokens y más formas distintas que el Quijote. Sin embargo, contar documentos por sí solo no refleja el tamaño de los datos: el único documento del Quijote contiene cientos de miles de tokens. Por eso, para comparar la cantidad de texto disponible para entrenar embeddings, resulta más útil observar los tokens y los contextos.

La salida confirma que ambos corpus quedaron como **listas de listas de tokens**. Se conservan palabras como `el`, `de` y `que`, así como las tildes de `andrés` y `garcía`. Esta representación mantiene el orden necesario para aprender de palabras vecinas. La asignación ilustrativa de IDs identifica cada token; los IDs todavía no son vectores ni expresan similitud semántica.

### Efecto de la frecuencia mínima

| Corpus | `min_count` | Tamaño del vocabulario | Tokens excluidos | Tokens excluidos (%) |
|---|---:|---:|---:|---:|
| Noticias | 1 | 29,507 | 0 | 0.0000 |
| Noticias | 2 | 17,129 | 12,378 | 1.9413 |
| Noticias | 5 | 8,890 | 34,319 | 5.3824 |
| Quijote | 1 | 22,943 | 0 | 0.0000 |
| Quijote | 2 | 11,760 | 11,183 | 2.9334 |
| Quijote | 5 | 5,301 | 28,121 | 7.3764 |

Con **`min_count=2`**, el vocabulario de noticias disminuye aproximadamente **41.95 %**, pero se conserva **98.06 % de las apariciones de tokens**. En el Quijote, el vocabulario se reduce **48.74 %** y se conserva **97.07 % de los tokens**. La diferencia entre ambos porcentajes muestra que muchas formas distintas aparecen una sola vez y representan una parte pequeña del texto total.

Elevar el umbral a **5** reduce aún más el vocabulario, aunque también excluye más apariciones: **5.38 %** de los tokens de noticias y **7.38 %** de los del Quijote. Esto puede dejar fuera palabras relevantes poco frecuentes, además de posibles errores o variantes raras. Por esa razón, **`min_count=2` es un punto de partida razonable** para el entrenamiento posterior: reduce considerablemente el vocabulario y mantiene una cobertura alta. Estos conteos todavía no permiten concluir qué umbral producirá mejores vecinos semánticos.

### Pares skip-gram y tamaño de ventana

La oración seleccionada fue:

> la sostenibilidad nos abre oportunidades a mejores fuentes de financiamiento agregó

Sus **11 tokens** producen **38 pares dirigidos** con ventana 2. Por ejemplo, `sostenibilidad` tiene como contextos `la`, `nos` y `abre`; su posición cercana al inicio limita la cantidad de vecinos disponibles. Una palabra interior como `oportunidades` tiene cuatro contextos: `nos`, `abre`, `a` y `mejores`.

Al aplicar el conteo a todas las oraciones de noticias se obtuvo:

| Ventana | Tokens del corpus | Pares de entrenamiento | Pares por token |
|---|---:|---:|---:|
| 2 | 637,610 | 2,419,436 | 3.7945 |
| 5 | 637,610 | 5,722,750 | 8.9753 |

La ventana 5 genera aproximadamente **2.37 veces** los pares de la ventana 2 sobre el mismo texto. Los promedios son menores que los máximos de **4** y **10** contextos por posición porque las palabras cercanas a los límites de oración tienen menos vecinos. Una ventana mayor incorpora palabras más alejadas y aumenta la cantidad de ejemplos potenciales, aunque estos resultados por sí solos no demuestran que mejore la calidad del modelo.

Word2Vec puede aprender sin categorías anotadas porque los ejemplos se obtienen del propio texto: cada palabra central sirve para predecir las palabras de su contexto. Las etiquetas de clasificación de noticias no intervienen en la generación de estos pares. Las cifras corresponden a una pasada con ventana fija, antes del filtrado por frecuencia y del submuestreo.

### Compatibilidad con fastText SUC

La carga del modelo se completó con **200,000 entradas y 100 dimensiones**. La consulta de las grafías exactas produjo estos resultados:

| Grafía | Presente en el vocabulario cargado |
|---|:---:|
| `españa` | Sí |
| `España` | No |
| `espana` | Sí |
| `ESPAÑA` | No |
| `tecnología` | Sí |
| `tecnologia` | Sí |
| `año` | Sí |
| `ano` | Sí |

Para el ejemplo consultado, la presencia de `españa` y la ausencia de `España` y `ESPAÑA` respaldan la conversión a minúsculas. Esta observación corresponde a las grafías probadas en la selección cargada; no demuestra que el archivo completo carezca de cualquier entrada con mayúsculas.

Las formas con y sin tilde están disponibles como claves diferentes. Su presencia no implica que sean equivalentes ni que debamos fusionarlas. Por eso se conservan las tildes y la `ñ`: `año` y `ano`, por ejemplo, son palabras distintas. Se consultará cada token con su grafía normalizada exacta en todos los modelos.

#### Cobertura de los corpus

| Corpus | Tipos cubiertos (%) | Tokens cubiertos (%) | Tokens fuera (%) |
|---|---:|---:|---:|
| Noticias | 86.10 | 98.70 | 1.30 |
| Quijote | 73.50 | 97.38 | 2.62 |

En noticias, SUC cubre **86.10 % de las formas distintas** y **98.70 % de las apariciones de tokens**. En el Quijote, cubre **73.50 % de los tipos** y **97.38 % de los tokens**. La cobertura de tokens es mayor que la de tipos en ambos casos: las formas que faltan concentran una proporción pequeña de las apariciones del texto.

El Quijote presenta menor cobertura que las noticias. Esto es compatible con diferencias de época, vocabulario y formas flexivas, aunque estas tablas no identifican cuáles de esas causas explican cada ausencia. Haría falta inspeccionar las palabras fuera del vocabulario para determinarlo.

Los resultados respaldan mantener la normalización propuesta y muestran una cobertura alta de los tokens de ambos corpus. En las comparaciones posteriores se reportarán los elementos fuera del vocabulario y se usarán consultas comunes cuando corresponda. Las ausencias se refieren a las **200,000 entradas cargadas**, no necesariamente a todo el archivo SUC.


# 2. Co-ocurrencia, PPMI y SVD

En esta sección se aplica la hipótesis distribucional al corpus de noticias: las palabras se representan mediante los contextos en los que aparecen. Primero se cuentan coocurrencias, después se ponderan con PPMI y finalmente se obtienen embeddings de 100 dimensiones mediante SVD.

Se reutilizan `oraciones_noticias` y `frecuencias_corpus` de la sección 1. Ejecutar primero esa preparación. Se requieren **NumPy**, **SciPy**, **scikit-learn** y **gensim**; si falta algún paquete, instalarlo en el kernel con `%pip install numpy scipy scikit-learn gensim`.

## 2.1 Matriz de coocurrencia palabra × palabra

Se seleccionan las **2,000 palabras más frecuentes**. Cada celda cuenta cuántas veces el término de su fila aparece con el de su columna en una ventana **±2**, con peso uniforme y sin atravesar límites de oración.

Las palabras fuera del vocabulario se excluyen del conteo, pero **conservan su posición** al medir distancias. Eliminarlas antes de construir las ventanas acercaría artificialmente otras palabras. Se excluye la propia posición central; dos posiciones distintas con la misma palabra sí pueden contribuir a la diagonal.

In [56]:
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

if "oraciones_noticias" not in globals() or "frecuencias_corpus" not in globals():
    raise RuntimeError("Ejecuta primero las celdas de preparación de la sección 1.")

TAMANO_VOCAB_COOC = 2_000
VENTANA_COOC = 2
vocabulario_cooc = [
    palabra for palabra, _ in frecuencias_corpus["Noticias"].most_common(TAMANO_VOCAB_COOC)
]
indice_cooc = {palabra: i for i, palabra in enumerate(vocabulario_cooc)}


def construir_coocurrencias(oraciones, indice, ventana=2):
    """Contar pares dirigidos conservando las posiciones de los tokens OOV."""
    matriz = np.zeros((len(indice), len(indice)), dtype=np.int64)
    for oracion in oraciones:
        ids = np.array([indice.get(token, -1) for token in oracion], dtype=np.int64)
        for distancia in range(1, min(ventana, len(ids) - 1) + 1):
            izquierda, derecha = ids[:-distancia], ids[distancia:]
            validos = (izquierda >= 0) & (derecha >= 0)
            a, b = izquierda[validos], derecha[validos]
            # Cada pareja de posiciones contribuye en las dos direcciones.
            np.add.at(matriz, (a, b), 1)
            np.add.at(matriz, (b, a), 1)
    return matriz


matriz_cooc = construir_coocurrencias(oraciones_noticias, indice_cooc, VENTANA_COOC)
no_ceros_cooc = int(np.count_nonzero(matriz_cooc))
ceros_cooc_pct = 100 * (1 - no_ceros_cooc / matriz_cooc.size)
resumen_cooc = pd.DataFrame([{
    "Forma": str(matriz_cooc.shape),
    "Celdas totales": matriz_cooc.size,
    "Celdas no nulas": no_ceros_cooc,
    "Ceros (%)": round(ceros_cooc_pct, 4),
    "Suma de conteos dirigidos": int(matriz_cooc.sum()),
}])
display(resumen_cooc)
print("¿La matriz es simétrica?", np.array_equal(matriz_cooc, matriz_cooc.T))

,Forma,Celdas totales,Celdas no nulas,Ceros (%),Suma de conteos dirigidos
0,"(2000, 2000)",4000000,252148,93.6963,1617360


¿La matriz es simétrica? True


### Comparación con BoW del Laboratorio 2

La ejecución guardada de `Lab2/lab2.ipynb` reporta una matriz BoW de **1,217 × 21,944**, con **230,088 celdas no nulas** y **99.1384 % de ceros**. Se utilizan sus conteos exactos como referencia.

BoW relaciona documentos con palabras; la matriz actual relaciona palabras con contextos locales. Además, el Laboratorio 2 retiró stopwords y aplicó lematización, mientras que aquí se conservan, y los tamaños de vocabulario son distintos. Por eso, la diferencia de dispersión describe estas dos representaciones concretas y no aísla el efecto del método.

In [57]:
# Referencia: salidas guardadas de las secciones 1 y 2 de Lab2/lab2.ipynb.
forma_bow_lab2 = (1_217, 21_944)
no_ceros_bow_lab2 = 230_088
ceros_bow_lab2_pct = 100 * (1 - no_ceros_bow_lab2 / np.prod(forma_bow_lab2))
comparacion_dispersion = pd.DataFrame([
    {"Representación": "BoW · Laboratorio 2", "Ejes": "Documentos × palabras",
     "Forma": str(forma_bow_lab2), "Celdas no nulas": no_ceros_bow_lab2,
     "Ceros (%)": ceros_bow_lab2_pct},
    {"Representación": "Coocurrencia · Laboratorio 9", "Ejes": "Palabras × palabras",
     "Forma": str(matriz_cooc.shape), "Celdas no nulas": no_ceros_cooc,
     "Ceros (%)": ceros_cooc_pct},
])
comparacion_dispersion["Ceros (%)"] = comparacion_dispersion["Ceros (%)"].round(4)
display(comparacion_dispersion)
print(f"Diferencia de ceros (coocurrencia − BoW): {ceros_cooc_pct - ceros_bow_lab2_pct:.4f} puntos porcentuales")

,Representación,Ejes,Forma,Celdas no nulas,Ceros (%)
0,BoW · Laboratorio 2,Documentos × palabras,"(1217, 21944)",230088,99.1384
1,Coocurrencia · Laboratorio 9,Palabras × palabras,"(2000, 2000)",252148,93.6963


Diferencia de ceros (coocurrencia − BoW): -5.4421 puntos porcentuales


## 2.2 Similitud coseno entre filas

Se comparan cinco pares elegidos antes de observar sus similitudes. La columna de expectativa expresa una intuición cualitativa, no una etiqueta de evaluación. El coseno compara el perfil completo de contextos de ambas palabras; no equivale a su frecuencia de aparición conjunta.

In [58]:
def coseno_conteos(u, v):
    u, v = np.asarray(u, dtype=np.float64), np.asarray(v, dtype=np.float64)
    denominador = np.linalg.norm(u) * np.linalg.norm(v)
    return float(np.dot(u, v) / denominador) if denominador else np.nan


pares_cooc = [
    ("banco", "entidad", "Relacionadas: instituciones financieras"),
    ("gobierno", "estado", "Relacionadas: ámbito político"),
    ("empresa", "compañía", "Relacionadas: organizaciones empresariales"),
    ("banco", "gobierno", "Relación temática: economía y políticas públicas"),
    ("empresa", "febrero", "Sin relación semántica directa evidente"),
]
filas_similitud = []
for palabra_a, palabra_b, expectativa in pares_cooc:
    faltantes = [p for p in (palabra_a, palabra_b) if p not in indice_cooc]
    valor = (coseno_conteos(matriz_cooc[indice_cooc[palabra_a]],
                            matriz_cooc[indice_cooc[palabra_b]])
             if not faltantes else np.nan)
    filas_similitud.append({
        "Palabra A": palabra_a, "Palabra B": palabra_b,
        "Expectativa": expectativa, "Coseno de conteos": valor,
        "Estado": "En vocabulario" if not faltantes else f"Fuera: {', '.join(faltantes)}",
    })
similitudes_cooc = pd.DataFrame(filas_similitud)
display(similitudes_cooc.round({"Coseno de conteos": 6}))

,Palabra A,Palabra B,Expectativa,Coseno de conteos,Estado
0,banco,entidad,Relacionadas: instituciones financieras,0.322416,En vocabulario
1,gobierno,estado,Relacionadas: ámbito político,0.779640,En vocabulario
2,empresa,compañía,Relacionadas: organizaciones empresariales,0.959576,En vocabulario
3,banco,gobierno,Relación temática: economía y políticas públicas,0.934576,En vocabulario
4,empresa,febrero,Sin relación semántica directa evidente,0.655571,En vocabulario


## 2.3 Contextos con mayor conteo

Se muestran los diez contextos de mayor frecuencia para `empresa` y `gobierno`. Este ranking utiliza directamente las celdas de cada fila; no busca palabras con filas similares. Se excluye la palabra consultada y se incluyen únicamente asociaciones con peso positivo. Los empates se resuelven por el orden del vocabulario.

In [59]:
def contextos_principales(matriz, palabra, cantidad=10):
    if palabra not in indice_cooc:
        raise KeyError(f"{palabra!r} no está entre las palabras seleccionadas.")
    fila = matriz[indice_cooc[palabra]]
    orden = np.argsort(-fila, kind="stable")
    elegidos = [j for j in orden
                if vocabulario_cooc[j] != palabra and fila[j] > 0][:cantidad]
    return [(vocabulario_cooc[j], float(fila[j])) for j in elegidos]


PALABRAS_CONSULTA = ("empresa", "gobierno")
contextos_conteo = pd.DataFrame([
    {"Consulta": palabra, "Posición": posicion, "Contexto": contexto,
     "Coocurrencias": int(peso)}
    for palabra in PALABRAS_CONSULTA
    for posicion, (contexto, peso) in enumerate(
        contextos_principales(matriz_cooc, palabra), start=1
    )
])
display(contextos_conteo)

,Consulta,Posición,Contexto,Coocurrencias
0,empresa,1,la,205
1,empresa,2,de,152
2,empresa,3,una,65
3,empresa,4,en,48
4,empresa,5,que,47
5,empresa,6,y,39
6,empresa,7,a,32
7,empresa,8,con,28
8,empresa,9,para,19
9,empresa,10,como,17


## 2.4 PPMI implementada con NumPy

La información mutua puntual compara la probabilidad observada de un par con la que tendría si palabra y contexto fueran independientes. PPMI conserva solamente los valores positivos:

$$\operatorname{PPMI}(w,c)=\max\left(0,\log_2\frac{P(w,c)}{P(w)P(c)}\right).$$

Con $X$ como matriz de conteos y $N=\sum_{w,c}X_{wc}$:

$$P(w,c)=\frac{X_{wc}}{N},\qquad P(w)=\frac{\sum_c X_{wc}}{N},\qquad P(c)=\frac{\sum_w X_{wc}}{N}.$$

Las probabilidades se estiman sobre los pares incluidos en la matriz restringida a 2,000 términos. Los pares nunca observados reciben valor cero. La implementación opera solamente sobre conteos positivos para evitar calcular $\log_2(0)$.

In [60]:
def calcular_ppmi(conteos):
    x = np.asarray(conteos, dtype=np.float64)
    total = x.sum()
    if total <= 0:
        raise ValueError("La matriz debe contener al menos una coocurrencia.")
    marginal_palabra = x.sum(axis=1)
    marginal_contexto = x.sum(axis=0)
    filas, columnas = np.nonzero(x)
    # log2(X_wc * N / (suma_fila * suma_columna)), sin logaritmos de cero.
    pmi = (np.log2(x[filas, columnas]) + np.log2(total)
           - np.log2(marginal_palabra[filas])
           - np.log2(marginal_contexto[columnas]))
    ppmi = np.zeros_like(x)
    ppmi[filas, columnas] = np.maximum(0.0, pmi)
    return ppmi


matriz_ppmi = calcular_ppmi(matriz_cooc)
resumen_ppmi = pd.DataFrame([{
    "Forma": str(matriz_ppmi.shape),
    "Celdas con PPMI positiva": int(np.count_nonzero(matriz_ppmi)),
    "Ceros (%)": round(100 * (1 - np.count_nonzero(matriz_ppmi) / matriz_ppmi.size), 4),
    "PPMI máxima": float(matriz_ppmi.max()),
}])
display(resumen_ppmi)

contextos_ppmi = pd.DataFrame([
    {"Consulta": palabra, "Posición": posicion, "Contexto": contexto, "PPMI": peso}
    for palabra in PALABRAS_CONSULTA
    for posicion, (contexto, peso) in enumerate(
        contextos_principales(matriz_ppmi, palabra), start=1
    )
])
comparacion_contextos = contextos_conteo.merge(
    contextos_ppmi, on=["Consulta", "Posición"], how="outer",
    suffixes=(" por conteo", " por PPMI"),
)
display(comparacion_contextos.round({"PPMI": 4}))

,Forma,Celdas con PPMI positiva,Ceros (%),PPMI máxima
0,"(2000, 2000)",202756,94.9311,12.512625


,Consulta,Posición,Contexto por conteo,Coocurrencias,Contexto por PPMI,PPMI
0,empresa,1,la,205,telecomunicaciones,5.4715
1,empresa,2,de,152,líder,5.1641
2,empresa,3,una,65,llega,5.0986
3,empresa,4,en,48,especializada,5.0256
4,empresa,5,que,47,cualquier,4.8479
5,empresa,6,y,39,estadounidense,4.6676
6,empresa,7,a,32,colombiana,4.5755
7,empresa,8,con,28,asegurar,4.5570
8,empresa,9,para,19,criterios,4.4611
9,empresa,10,como,17,española,4.2625


PPMI reduce el peso de coincidencias explicables por la frecuencia general de sus términos y destaca asociaciones que ocurren más de lo esperado bajo independencia. Esto puede cambiar los contextos mejor posicionados. También puede favorecer asociaciones raras; un valor alto no garantiza por sí solo sinonimia ni una relación estable.

## 2.5 Reducción a 100 dimensiones con SVD

Se aplica `TruncatedSVD` a la matriz PPMI con una semilla fija. Cada fila del resultado es un vector de 100 dimensiones asociado a la palabra de la misma posición en `vocabulario_cooc`. Se utiliza la representación $U_{100}\Sigma_{100}$ obtenida mediante `fit_transform`, sin centrar las columnas.

In [61]:
DIMENSION_SVD = 100
svd_ppmi = TruncatedSVD(n_components=DIMENSION_SVD, n_iter=10, random_state=42)
vectores_svd = svd_ppmi.fit_transform(csr_matrix(matriz_ppmi)).astype(np.float32)

resumen_svd = pd.DataFrame([{
    "Palabras": vectores_svd.shape[0],
    "Dimensión original": matriz_ppmi.shape[1],
    "Dimensión reducida": vectores_svd.shape[1],
    "Varianza explicada acumulada (%)": 100 * svd_ppmi.explained_variance_ratio_.sum(),
}])
display(resumen_svd.round({"Varianza explicada acumulada (%)": 4}))
print("¿Todos los valores son finitos?", bool(np.isfinite(vectores_svd).all()))

# Ejemplo de las primeras diez coordenadas para las palabras consultadas.
display(pd.DataFrame(
    [vectores_svd[indice_cooc[palabra], :10] for palabra in PALABRAS_CONSULTA],
    index=pd.Index(PALABRAS_CONSULTA, name="Palabra"),
    columns=[f"Dimensión {i}" for i in range(1, 11)],
).round(4))

,Palabras,Dimensión original,Dimensión reducida,Varianza explicada acumulada (%)
0,2000,2000,100,25.4955


¿Todos los valores son finitos? True


,Dimensión 1,Dimensión 2,Dimensión 3,Dimensión 4,Dimensión 5,Dimensión 6,Dimensión 7,Dimensión 8,Dimensión 9,Dimensión 10
Palabra,,,,,,,,,,
empresa,8.6317,-2.7268,-4.4077,0.4813,-2.9925,-0.9075,1.0791,-3.1626,-0.7511,4.6180
gobierno,9.9422,-1.2800,-4.5500,1.8302,1.2317,-0.6604,-1.7739,3.2670,-1.0352,-0.9727


### Guardar los embeddings como KeyedVectors

Los vectores se guardan en `Lab9/ppmi_svd_100.kv` para cargarlos en la sección 5 mediante `KeyedVectors.load`. Esta celda requiere gensim, pero **no necesita el archivo SUC**. Se comprueba que una palabra conserve su vector después de volver a abrir el archivo.

#### Dependencia para exportar los vectores

Si aparece `ModuleNotFoundError: No module named 'gensim'`, ejecuta la siguiente celda. `%pip` instala el paquete en el entorno del kernel de este notebook. Después vuelve a ejecutar la celda de exportación; no es necesario repetir el cálculo de SVD si sus variables siguen en memoria.


In [62]:
import importlib.util

if importlib.util.find_spec("gensim") is None:
    %pip install gensim
else:
    print("gensim ya está instalado en este kernel.")


gensim ya está instalado en este kernel.


In [63]:
from gensim.models import KeyedVectors

svd_kv = KeyedVectors(vector_size=DIMENSION_SVD)
svd_kv.add_vectors(vocabulario_cooc, vectores_svd)
RUTA_SVD_KV = RAIZ / "Lab9/ppmi_svd_100.kv"
svd_kv.save(str(RUTA_SVD_KV), separately=[])

svd_kv_recargado = KeyedVectors.load(str(RUTA_SVD_KV))
assert svd_kv_recargado.index_to_key == vocabulario_cooc
np.testing.assert_allclose(svd_kv_recargado["empresa"], svd_kv["empresa"])
print(f"Archivo guardado: {RUTA_SVD_KV.relative_to(RAIZ)}")
print(f"Verificación de carga: {len(svd_kv_recargado):,} palabras × {svd_kv_recargado.vector_size} dimensiones")

Archivo guardado: Lab9/ppmi_svd_100.kv
Verificación de carga: 2,000 palabras × 100 dimensiones


## 2.6 Interpretación de las representaciones

Las filas de la matriz de coocurrencia ya son vectores de palabras porque cada una expresa numéricamente el perfil de contextos de un término. Sus coordenadas tienen una interpretación directa: cada dimensión corresponde a una palabra del vocabulario y registra la frecuencia con que aparece cerca de la palabra representada.

**PPMI cambia la ponderación:** permite distinguir una asociación específica de otra que aparece muchas veces simplemente porque sus palabras son frecuentes. Mantiene las 2,000 dimensiones originales y puede aumentar la cantidad de ceros al descartar asociaciones con PMI no positiva.

**SVD cambia la representación:** comprime los perfiles de PPMI en 100 dimensiones latentes que combinan información de varios contextos. Reduce el espacio de almacenamiento de los vectores y aproxima los patrones principales de la matriz, a cambio de perder información y la interpretación directa de cada coordenada.

### Análisis de los resultados

#### Dispersión de la matriz

La matriz de coocurrencia tiene **2,000 × 2,000 posiciones**, de las cuales **252,148 contienen conteos distintos de cero**. Su dispersión es **93.6963 %** y la suma de sus conteos dirigidos es **1,617,360**. La comprobación de simetría fue positiva, como corresponde a una ventana que cuenta los contextos a ambos lados con el mismo peso.

| Representación | Forma | Celdas no nulas | Ceros (%) |
|---|---:|---:|---:|
| BoW del Laboratorio 2 | 1,217 × 21,944 | 230,088 | 99.1384 |
| Coocurrencia de noticias | 2,000 × 2,000 | 252,148 | 93.6963 |

La matriz actual presenta **5.4421 puntos porcentuales menos de ceros** que BoW. Cada fila acumula los contextos de una palabra a lo largo de todo el corpus y el vocabulario se limita a términos frecuentes. Sin embargo, también cambiaron el preprocesamiento, las dimensiones y el significado de los ejes; no se puede atribuir toda la diferencia únicamente al uso de coocurrencias.

#### ¿Los cosenos coinciden con la intuición?

| Par de palabras | Coseno de conteos | Interpretación |
|---|---:|---|
| banco – entidad | 0.322416 | Es el valor más bajo, aunque esperábamos una relación financiera. |
| gobierno – estado | 0.779640 | Es compatible con una relación temática política. |
| empresa – compañía | 0.959576 | Es el valor más alto y coincide con la cercanía esperada. |
| banco – gobierno | 0.934576 | Comparten perfiles de contexto muy parecidos, aunque no son sinónimos. |
| empresa – febrero | 0.655571 | La similitud es considerable pese a no tener una relación semántica directa evidente. |

Los resultados coinciden **solo parcialmente** con la intuición. `empresa` y `compañía` presentan perfiles casi paralelos, pero `banco` y `entidad` no muestran la misma cercanía. Una posible explicación es que `entidad` se utilice con un sentido más amplio; confirmarlo requeriría revisar sus contextos específicos.

El valor de `empresa–febrero` muestra que el coseno sobre conteos crudos no mide directamente sinonimia. Las palabras funcionales muy frecuentes pueden influir mucho en la orientación de los vectores. Los contextos principales observados para `empresa` y `gobierno` respaldan esa limitación de los conteos, aunque no permiten aislar por sí solos la causa de cada coseno.

#### ¿Qué cambia al aplicar PPMI?

En los conteos de `empresa` dominan artículos, preposiciones y otras palabras funcionales: `la` aparece **205** veces y `de`, **152**. Para `gobierno` destacan `el`, con **218**, y `del`, con **108**. Entre sus diez contextos principales, `nacional` es el término de contenido más evidente para `gobierno`.

PPMI cambia el orden al considerar también la frecuencia marginal de cada término:

| Posición | Contexto de empresa por conteo | Contexto de empresa por PPMI | Contexto de gobierno por conteo | Contexto de gobierno por PPMI |
|---:|---|---|---|---|
| 1 | la | telecomunicaciones | el | corporativo |
| 2 | de | líder | del | federal |
| 3 | una | llega | de | sánchez |
| 4 | en | especializada | que | nacional |
| 5 | que | cualquier | y | iván |
| 6 | y | estadounidense | nacional | funcionarios |
| 7 | a | colombiana | en | estará |
| 8 | con | asegurar | para | petro |
| 9 | para | criterios | por | duque |
| 10 | como | española | al | pueda |

La asociación mejor posicionada de `empresa` pasa a ser `telecomunicaciones`, con **PPMI = 5.4715**; para `gobierno` es `corporativo`, con **PPMI = 7.2663**. Aparecen sectores, características de empresas, nombres propios y expresiones como *gobierno corporativo* o *gobierno federal*. Estas asociaciones reflejan el dominio del corpus y no necesariamente relaciones de sinonimia.

Las palabras funcionales pierden protagonismo porque su alta frecuencia general hace que muchas de sus coincidencias sean esperables incluso bajo independencia. PPMI destaca encuentros más específicos, pero también conserva resultados como `llega`, `estará` o `pueda`: la ponderación no garantiza que todos los contextos sean descriptores temáticos.

Después de transformar los conteos, quedan **202,756 celdas con PPMI positiva** y **94.9311 % de ceros**. La dispersión aumenta porque las asociaciones con PMI no positiva se llevan a cero. Las asociaciones raras también pueden obtener valores altos, por lo que conviene interpretar los rankings junto con sus frecuencias.

#### Resultado de SVD y archivo de vectores

SVD transforma cada perfil de **2,000 contextos** en un vector de **100 dimensiones**. El resultado tiene forma **2,000 × 100**, contiene únicamente valores finitos y explica **25.4955 % de la varianza**, según la medida reportada por scikit-learn. Este porcentaje describe la representación numérica y no una exactitud semántica.

La reducción utiliza **20 veces menos coordenadas por palabra**. Cada dimensión combina información de varios contextos, por lo que deja de corresponder a una palabra concreta. Esa compresión permite trabajar con vectores densos, aunque también descarta parte de la información original.

Los embeddings se guardaron en **`Lab9/ppmi_svd_100.kv`**. La recarga confirmó las **2,000 palabras**, las **100 dimensiones**, el orden del vocabulario y la conservación del vector de `empresa`. El archivo queda disponible para la comparación de modelos de la sección 5.


# 3. Investigación

**CBOW y skip-gram.** Al investigar, encontré que CBOW predice la palabra central combinando sus contextos; skip-gram predice cada contexto desde la central. Por eso, espero que skip-gram represente mejor palabras poco frecuentes al obtener varias predicciones por aparición, aunque depende de los datos y parámetros. [Word2Vec](https://arxiv.org/abs/1301.3781).

**Negative sampling y dos tablas.** Encontré que el softmax completo normaliza sobre todo el vocabulario. Entendí que negative sampling reduce ese costo clasificando pares observados frente a unos pocos pares de ruido:

$$P(D=1\mid w,c)=\sigma(v_w^\top c_c)=\frac{1}{1+e^{-v_w^\top c_c}}.$$

Identifiqué $v_w$ como el vector de la palabra central y $c_c$ como el del contexto. Interpreté la probabilidad como una clasificación de datos frente a ruido, no como una distribución normalizada sobre contextos. Comprendí que las dos tablas permiten aprender representaciones independientes para los roles de entrada y salida; normalmente conservaré la primera, disponible como `model.wv` en gensim. [Objetivo de entrenamiento](https://arxiv.org/html/1310.4546v1), [gensim](https://radimrehurek.com/gensim/models/word2vec.html).

**Word2Vec, GloVe y fastText.** Al compararlos, encontré que Word2Vec aprende mediante predicciones locales; GloVe ajusta vectores usando conteos globales de coocurrencia. FastText incorpora n-gramas de caracteres, comparte información entre formas y permite generar vectores para palabras no vistas. Considero útil este mecanismo para el español por sus conjugaciones y variaciones de género y número. [GloVe](https://aclanthology.org/D14-1162/), [fastText](https://aclanthology.org/Q17-1010/).

**Parámetros de Word2Vec.** Identifiqué estas funciones: `vector_size`: dimensiones; `window`: distancia máxima de contexto; `min_count`: frecuencia mínima; `sg`: 0 para CBOW y 1 para skip-gram; `negative`: muestras de ruido por ejemplo positivo; `sample`: umbral de submuestreo de palabras frecuentes; `epochs`: pasadas sobre el corpus. [Documentación](https://radimrehurek.com/gensim/models/word2vec.html).

**Capas de PyTorch.** Encontré que `nn.Embedding` almacena una tabla: `num_embeddings` indica sus filas y `embedding_dim`, las dimensiones; `padding_idx` identifica la fila de relleno, que no recibe gradientes. `from_pretrained` inicializa desde vectores existentes: `freeze=True` los congela y `False` permite ajustarlos. `nn.EmbeddingBag` agrega los vectores de cada secuencia con `mode="sum"`, `"mean"` o `"max"`, sin materializar todos los embeddings intermedios. Entendí que multiplicar el vector fila one-hot $e_i$ por $E$ selecciona su fila $i$: $e_iE=E_{i,:}$, porque las demás filas se multiplican por cero. [Embedding](https://docs.pytorch.org/docs/2.14/generated/torch.nn.Embedding.html), [EmbeddingBag](https://docs.pytorch.org/docs/2.14/generated/torch.nn.EmbeddingBag.html).


# 4. Entrenar Word2Vec y fastText

Para esta sección definí un modelo base y cinco variantes de Word2Vec. Comparé cómo cambiaron los vecinos al modificar la arquitectura, la ventana o la dimensión. Después contrasté los dominios de noticias y Quijote y probé palabras desconocidas con FastText.

Reutilizo los corpus de la sección 1. Esta sección **no requiere volver a cargar SUC**: el FastText que entrené aquí es un modelo propio de noticias. Conservé las salidas para fundamentar mis conclusiones.

## 4.1 Configuración y palabras de control

Fijé `seed=42`, `workers=1` y una función de hash estable para reducir variaciones entre ejecuciones. Todos los modelos Word2Vec comparten los parámetros que no están siendo comparados. Utilizo **10 épocas** y `sample=0.001`; mantengo el ajuste habitual de ventanas aleatorias de gensim (`shrink_windows=True`). El conteo de pares con ventana fija de la sección 1 es una referencia, no el conteo efectivo de este entrenamiento.

Consulté la documentación de [Word2Vec](https://radimrehurek.com/gensim/models/word2vec.html) y [FastText](https://radimrehurek.com/gensim/models/fasttext.html) para definir los parámetros y las consultas.

In [64]:
from time import perf_counter
import zlib
import numpy as np
import pandas as pd
from gensim.models import Word2Vec, FastText, KeyedVectors

if not all(nombre in globals() for nombre in
           ("oraciones_noticias", "oraciones_quijote", "frecuencias_corpus", "RAIZ")):
    raise RuntimeError("Ejecuta primero la preparación de la sección 1.")


def hash_estable(texto):
    return zlib.crc32(texto.encode("utf-8"))


CONFIG_BASE_W2V = dict(
    vector_size=100, window=5, min_count=2, sg=1, negative=5,
    epochs=10, sample=0.001, seed=42, workers=1,
    hs=0, shrink_windows=True, hashfxn=hash_estable,
)
CONFIGURACIONES_W2V = {
    "base": CONFIG_BASE_W2V.copy(),
    "cbow": {**CONFIG_BASE_W2V, "sg": 0},
    "ventana_2": {**CONFIG_BASE_W2V, "window": 2},
    "ventana_10": {**CONFIG_BASE_W2V, "window": 10},
    "dimension_50": {**CONFIG_BASE_W2V, "vector_size": 50},
    "dimension_300": {**CONFIG_BASE_W2V, "vector_size": 300},
}

candidatas_control = [
    "banco", "gobierno", "empresa", "mercado", "tecnología", "sostenibilidad",
    "economía", "financiero", "inversión", "desarrollo", "sector", "negocio",
]
vocabulario_entrenable = {
    palabra for palabra, frecuencia in frecuencias_corpus["Noticias"].items()
    if frecuencia >= CONFIG_BASE_W2V["min_count"]
}
PALABRAS_CONTROL = [p for p in candidatas_control if p in vocabulario_entrenable][:6]
if len(PALABRAS_CONTROL) != 6:
    raise ValueError("Revisa las palabras de control: deben ser seis términos entrenables.")
print("Palabras de control fijas:", PALABRAS_CONTROL)

tabla_configuraciones = pd.DataFrame([
    {"Modelo": nombre, **{k: config[k] for k in
     ("vector_size", "window", "min_count", "sg", "negative", "sample", "epochs")}}
    for nombre, config in CONFIGURACIONES_W2V.items()
])
display(tabla_configuraciones)

Palabras de control fijas: ['banco', 'gobierno', 'empresa', 'mercado', 'tecnología', 'sostenibilidad']


,Modelo,vector_size,window,min_count,sg,negative,sample,epochs
0,base,100,5,2,1,5,0.001,10
1,cbow,100,5,2,0,5,0.001,10
2,ventana_2,100,2,2,1,5,0.001,10
3,ventana_10,100,10,2,1,5,0.001,10
4,dimension_50,50,5,2,1,5,0.001,10
5,dimension_300,300,5,2,1,5,0.001,10


## 4.2 Entrenamiento del modelo base y sus variantes

Medí por separado el tiempo de construcción del vocabulario y el de entrenamiento. El total incluye ambas operaciones; no incluye las consultas de vecinos. Al volver a ejecutar esta celda se entrenan nuevamente las seis configuraciones.

In [65]:
def entrenar_embeddings(clase, oraciones, configuracion, **extras):
    inicio = perf_counter()
    modelo = clase(**configuracion, **extras)
    modelo.build_vocab(corpus_iterable=oraciones)
    fin_vocabulario = perf_counter()
    modelo.train(
        corpus_iterable=oraciones,
        total_examples=modelo.corpus_count,
        epochs=configuracion["epochs"],
    )
    fin = perf_counter()
    tiempos = {
        "Vocabulario (s)": fin_vocabulario - inicio,
        "Entrenamiento (s)": fin - fin_vocabulario,
        "Total (s)": fin - inicio,
    }
    return modelo, tiempos


modelos_w2v_noticias = {}
registros_entrenamiento = []
for nombre, configuracion in CONFIGURACIONES_W2V.items():
    print(f"Entrenando Word2Vec: {nombre}...", flush=True)
    modelo, tiempos = entrenar_embeddings(Word2Vec, oraciones_noticias, configuracion)
    modelos_w2v_noticias[nombre] = modelo
    registros_entrenamiento.append({
        "Modelo": nombre, "Palabras": len(modelo.wv), **tiempos,
    })
    print(f"  Finalizado: {tiempos['Entrenamiento (s)']:.2f} s de entrenamiento.", flush=True)

tabla_entrenamiento_w2v = tabla_configuraciones.merge(
    pd.DataFrame(registros_entrenamiento), on="Modelo", validate="one_to_one"
)
display(tabla_entrenamiento_w2v.round(3))

Entrenando Word2Vec: base...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


  Finalizado: 10.31 s de entrenamiento.
Entrenando Word2Vec: cbow...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


  Finalizado: 3.16 s de entrenamiento.
Entrenando Word2Vec: ventana_2...


...

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


  Finalizado: 8.61 s de entrenamiento.
Entrenando Word2Vec: dimension_300...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


  Finalizado: 20.38 s de entrenamiento.


,Modelo,vector_size,window,min_count,sg,negative,sample,epochs,Palabras,Vocabulario (s),Entrenamiento (s),Total (s)
0,base,100,5,2,1,5,0.001,10,17129,0.169,10.315,10.483
1,cbow,100,5,2,0,5,0.001,10,17129,0.125,3.161,3.287
2,ventana_2,100,2,2,1,5,0.001,10,17129,0.121,5.754,5.874
3,ventana_10,100,10,2,1,5,0.001,10,17129,0.118,17.321,17.438
4,dimension_50,50,5,2,1,5,0.001,10,17129,0.119,8.614,8.733
5,dimension_300,300,5,2,1,5,0.001,10,17129,0.133,20.379,20.511


## 4.3 Cinco vecinos para las seis palabras de control

Comparé cada variante sobre el mismo conjunto de palabras. Muestro las cinco vecinas y sus cosenos en cada fila, junto con la configuración y el tiempo. Revisé si las ventanas pequeñas favorecían formas o usos sintácticos parecidos y si las grandes reunían términos temáticos; contrasté esa expectativa con los vecinos obtenidos.

In [66]:
filas_vecinos_w2v = []
for nombre, modelo in modelos_w2v_noticias.items():
    for palabra in PALABRAS_CONTROL:
        vecinos = modelo.wv.most_similar(palabra, topn=5)
        fila = {"Modelo": nombre, "Palabra de control": palabra}
        fila.update({f"Vecino {i}": f"{vecino} ({coseno:.4f})"
                     for i, (vecino, coseno) in enumerate(vecinos, 1)})
        filas_vecinos_w2v.append(fila)
vecinos_w2v_noticias = pd.DataFrame(filas_vecinos_w2v)
tabla_variantes_vecinos = vecinos_w2v_noticias.merge(
    tabla_entrenamiento_w2v, on="Modelo", validate="many_to_one"
)
columnas_variantes = [
    "Modelo", "vector_size", "window", "min_count", "sg", "negative", "sample",
    "epochs", "Entrenamiento (s)", "Palabra de control",
    *[f"Vecino {i}" for i in range(1, 6)],
]
with pd.option_context("display.max_rows", None, "display.max_columns", None,
                       "display.max_colwidth", None):
    display(tabla_variantes_vecinos[columnas_variantes].round({"Entrenamiento (s)": 3}))

# Comparación directa de ventanas, manteniendo las demás condiciones.
comparacion_ventanas = vecinos_w2v_noticias[
    vecinos_w2v_noticias["Modelo"].isin(["ventana_2", "ventana_10"])
]
print("Comparación de ventana 2 frente a ventana 10:")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(comparacion_ventanas)

,Modelo,vector_size,window,min_count,sg,negative,sample,epochs,Entrenamiento (s),Palabra de control,Vecino 1,Vecino 2,Vecino 3,Vecino 4,Vecino 5
0,base,100,5,2,1,5,0.001,10,10.315,banco,central (0.7007),popular (0.6011),república (0.5931),agrario (0.5663),davivienda (0.5465)
1,base,100,5,2,1,5,0.001,10,10.315,gobierno,congreso (0.7079),electo (0.6283),consejo (0.6189),gremial (0.6163),mandatario (0.6091)
2,base,100,5,2,1,5,0.001,10,10.315,empresa,startup (0.7208),compañía (0.6905),certificación (0.6560),emergente (0.6449),monitoreo (0.6421)
3,base,100,5,2,1,5,0.001,10,10.315,mercado,colombiano (0.5599),sector (0.5350),apetito (0.5199),accionario (0.5073),hueco (0.5048)
4,base,100,5,2,1,5,0.001,10,10.315,tecnología,estandarización (0.6801),computación (0.6659),cuántica (0.6564),tecnológica (0.6535),utilizará (0.6505)
5,base,100,5,2,1,5,0.001,10,10.315,sostenibilidad,digitalización (0.6481),diversidad (0.6374),alineado (0.6332),fundación (0.6222),prioridades (0.6134)
6,cbow,100,5,2,0,5,0.001,10,3.161,banco,cordillera (0.6776),grupo (0.5591),gradualidad (0.5146),equipo (0.5062),área (0.5028)
7,cbow,100,5,2,0,5,0.001,10,3.161,gobierno,congreso (0.7616),estado (0.7510),consejo (0.7200),organismo (0.7030),coronavirus (0.6897)
8,cbow,100,5,2,0,5,0.001,10,3.161,empresa,plataforma (0.8654),marca (0.8468),compañía (0.8383),iniciativa (0.8235),startup (0.8181)
9,cbow,100,5,2,0,5,0.001,10,3.161,mercado,país (0.7720),peso (0.7145),sector (0.7043),empleo (0.7030),territorio (0.6975)


Comparación de ventana 2 frente a ventana 10:


,Modelo,Palabra de control,Vecino 1,Vecino 2,Vecino 3,Vecino 4,Vecino 5
12,ventana_2,banco,central (0.5495),gnb (0.5409),bce (0.5267),popular (0.5250),sudameris (0.5218)
13,ventana_2,gobierno,congreso (0.7479),bce (0.6841),federal (0.6669),consejo (0.6430),senado (0.6265)
14,ventana_2,empresa,startup (0.7502),compañía (0.7460),jurisdicción (0.6794),criptomoneda (0.6752),película (0.6677)
15,ventana_2,mercado,sector (0.6304),panorama (0.4964),fútbol (0.4947),sistema (0.4944),dificultan (0.4926)
16,ventana_2,tecnología,publicidad (0.7257),computación (0.6969),capacitación (0.6966),educativa (0.6942),funcionalidad (0.6923)
17,ventana_2,sostenibilidad,digitalización (0.7706),emprendimiento (0.7151),creatividad (0.7137),transformación (0.7038),liderazgo (0.6913)
18,ventana_10,banco,central (0.6905),agrario (0.6035),popular (0.5714),república (0.5708),caja (0.5566)
19,ventana_10,gobierno,congreso (0.6828),detalló (0.6763),biden (0.6468),intervenir (0.6390),pacic (0.6358)
20,ventana_10,empresa,compañía (0.6413),enfocada (0.6277),startup (0.6179),consentio (0.6162),vincent (0.6059)
21,ventana_10,mercado,colombiano (0.6002),accionario (0.5184),apetito (0.5060),bursátil (0.5021),jugador (0.5011)


## 4.4 El significado según el corpus: noticias y Quijote

Entrené Word2Vec sobre el Quijote con exactamente la configuración base de noticias. Comparé `señor`, `dinero`, `casa`, `palabra` y `trabajo`, presentes en ambos vocabularios. Reemplacé `caballo` por `trabajo` porque no tenía cobertura común tras aplicar `min_count=2`. Así pude consultar cinco palabras sin comparar una palabra conocida con otra fuera del vocabulario.


In [67]:
print("Entrenando Word2Vec del Quijote...", flush=True)
modelo_w2v_quijote, tiempos_quijote = entrenar_embeddings(
    Word2Vec, oraciones_quijote, CONFIG_BASE_W2V
)
modelo_w2v_noticias_base = modelos_w2v_noticias["base"]
comunes = set(modelo_w2v_quijote.wv.key_to_index) & set(modelo_w2v_noticias_base.wv.key_to_index)
candidatas_dominio = [
    "señor", "dinero", "casa", "caballo", "palabra", "trabajo", "mundo",
    "tiempo", "persona", "hombre", "vida", "país", "ciudad", "poder",
]
PALABRAS_DOMINIO = [p for p in candidatas_dominio if p in comunes][:5]
if len(PALABRAS_DOMINIO) != 5:
    raise ValueError("Debes seleccionar cinco palabras compartidas por ambos vocabularios.")
print("Palabras seleccionadas:", PALABRAS_DOMINIO)
print("Ejemplos sugeridos sin cobertura común:",
      [p for p in candidatas_dominio[:5] if p not in comunes])
display(pd.DataFrame([{"Corpus": "Quijote", "Palabras": len(modelo_w2v_quijote.wv),
                       **tiempos_quijote}]).round(3))

filas_dominio = []
for palabra in PALABRAS_DOMINIO:
    fila = {"Palabra": palabra}
    for dominio, modelo in [("Noticias", modelo_w2v_noticias_base),
                            ("Quijote", modelo_w2v_quijote)]:
        fila[f"Vecinos de {dominio}"] = "; ".join(
            f"{vecino} ({coseno:.4f})"
            for vecino, coseno in modelo.wv.most_similar(palabra, topn=5)
        )
    filas_dominio.append(fila)
comparacion_dominios = pd.DataFrame(filas_dominio)
with pd.option_context("display.max_colwidth", None):
    display(comparacion_dominios)

Entrenando Word2Vec del Quijote...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Palabras seleccionadas: ['señor', 'dinero', 'casa', 'palabra', 'trabajo']
Ejemplos sugeridos sin cobertura común: ['caballo']


,Corpus,Palabras,Vocabulario (s),Entrenamiento (s),Total (s)
0,Quijote,11760,0.117,6.185,6.303


,Palabra,Vecinos de Noticias,Vecinos de Quijote
0,señor,encuadernador (0.8946); sustentabilidad (0.8941); unimos (0.8929); arma (0.8923); lempa (0.8921),hermano (0.6685); álvaro (0.6599); castellano (0.6530); tarfe (0.6526); recio (0.6474)
1,dinero,vivir (0.7080); estorbo (0.6843); crediticio (0.6823); virtuoso (0.6662); click (0.6630),arráez (0.8421); hallazgo (0.8356); dueño (0.8353); tema (0.8317); haréis (0.8266)
2,casa,pasiva (0.6573); caminar (0.6197); teletrabajo (0.6158); filtro (0.6133); compatibilizar (0.6046),aldea (0.6878); corte (0.6128); mujer (0.5987); hacienda (0.5966); majestad (0.5959)
3,palabra,sucedería (0.8776); inevitable (0.8734); viable (0.8728); fama (0.8725); huir (0.8695),hablar (0.7280); respondelle (0.7099); hablase (0.6980); responder (0.6873); persuadió (0.6872)
4,trabajo,perfil (0.6022); remoto (0.5844); colaborativo (0.5809); teletrabajo (0.5746); analíticos (0.5362),recogimiento (0.7777); descanso (0.7706); alivio (0.7566); daño (0.7549); igualar (0.7541)


## 4.5 FastText propio y palabras fuera del vocabulario

Utilicé la misma configuración base de entrenamiento. Para las subpalabras fijé `min_n=3`, `max_n=6` y `bucket=200_000`; este último limita la memoria de la tabla de n-gramas, a costa de posibles colisiones de hash. Es un ajuste adicional de FastText y lo mantuve explícito al describir el experimento.

Elegí tres palabras ausentes **incluso del texto tokenizado de noticias**, no solo de los términos descartados por `min_count`: un error de dedo, una conjugación poco común y una palabra inventada. Comprobé la ausencia exacta con `key_to_index`, porque FastText puede responder a consultas de términos que no tiene registrados como palabras completas.

In [68]:
print("Entrenando FastText de noticias...", flush=True)
modelo_fasttext_noticias, tiempos_fasttext = entrenar_embeddings(
    FastText, oraciones_noticias, CONFIG_BASE_W2V,
    min_n=3, max_n=6, bucket=200_000,
)
display(pd.DataFrame([{
    "Modelo": "FastText propio", "Palabras": len(modelo_fasttext_noticias.wv),
    "Dimensiones": modelo_fasttext_noticias.vector_size,
    "min_n": 3, "max_n": 6, "bucket": 200_000, **tiempos_fasttext,
}]).round(3))

candidatas_oov = {
    "Error de dedo": ["gobiernno", "sostenibiliddad", "tecnologíia"],
    "Conjugación poco común": ["gobernásemos", "gobernaseis", "financiásemos"],
    "Palabra inventada": ["criptoempresa", "neurofinanzópolis", "hipercriptoempresa"],
}
consultas_oov = {}
for tipo, candidatas in candidatas_oov.items():
    palabra = next((p for p in candidatas
                    if p not in frecuencias_corpus["Noticias"]
                    and p not in modelo_w2v_noticias_base.wv.key_to_index
                    and p not in modelo_fasttext_noticias.wv.key_to_index), None)
    if palabra is None:
        raise ValueError(f"No encontré un ejemplo realmente desconocido para: {tipo}")
    consultas_oov[tipo] = palabra

vectores_oov_fasttext = {}
filas_oov = []
for tipo, palabra in consultas_oov.items():
    vector = modelo_fasttext_noticias.wv.get_vector(palabra).copy()
    vectores_oov_fasttext[palabra] = vector  # Conservo las 100 coordenadas.
    vecinos = modelo_fasttext_noticias.wv.most_similar(positive=[vector], topn=5)
    try:
        modelo_w2v_noticias_base.wv.get_vector(palabra)
        resultado_w2v = "Vector disponible"
    except KeyError:
        resultado_w2v = "KeyError: palabra fuera del vocabulario"
    filas_oov.append({
        "Tipo": tipo, "Palabra": palabra,
        "Apariciones en noticias": frecuencias_corpus["Noticias"].get(palabra, 0),
        "En vocabulario FastText": palabra in modelo_fasttext_noticias.wv.key_to_index,
        "Dimensiones FastText": vector.size,
        "Norma del vector": float(np.linalg.norm(vector)),
        "Primeras 8 coordenadas": np.round(vector[:8], 4).tolist(),
        "Vecinos FastText": "; ".join(f"{w} ({s:.4f})" for w, s in vecinos),
        "Resultado Word2Vec": resultado_w2v,
    })
resultados_oov = pd.DataFrame(filas_oov)
with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(resultados_oov)

Entrenando FastText de noticias...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


,Modelo,Palabras,Dimensiones,min_n,max_n,bucket,Vocabulario (s),Entrenamiento (s),Total (s)
0,FastText propio,17129,100,3,6,200000,0.533,23.133,23.666


,Tipo,Palabra,Apariciones en noticias,En vocabulario FastText,Dimensiones FastText,Norma del vector,Primeras 8 coordenadas,Vecinos FastText,Resultado Word2Vec
0,Error de dedo,gobiernno,0,False,100,2.862251,"[0.6104000210762024, 0.3944000005722046, 0.5907999873161316, 0.7630000114440918, -0.42480000853538513, 0.04470000043511391, -0.47920000553131104, -0.16619999706745148]",gobierno (0.9724); gobiernos (0.8827); niinisto (0.7460); gobernar (0.7146); gobernador (0.7051),KeyError: palabra fuera del vocabulario
1,Conjugación poco común,gobernásemos,0,False,100,2.129152,"[0.23420000076293945, 0.20090000331401825, 0.4004000127315521, 0.09179999679327011, -0.09080000221729279, -0.20829999446868896, -0.09149999916553497, -0.07720000296831131]",decidimos (0.8565); coincidimos (0.8559); reforzamos (0.8499); andemos (0.8486); anónimos (0.8465),KeyError: palabra fuera del vocabulario
2,Palabra inventada,criptoempresa,0,False,100,2.868378,"[0.4625000059604645, 0.4185999929904938, 0.47690001130104065, 0.035100001841783524, -0.4909000098705292, 0.08349999785423279, -0.1890999972820282, -0.3043000102043152]",empresa (0.8388); criptomoneda (0.7808); criptografía (0.7746); cripto (0.7650); impresora (0.7622),KeyError: palabra fuera del vocabulario


La diferencia que observé fue que FastText construyó vectores para las tres palabras desconocidas a partir de sus subpalabras, mientras Word2Vec produjo `KeyError` al no tener entradas entrenadas para ellas. Encontré resultados plausibles para el error de dedo y la palabra inventada, pero la conjugación mostró una limitación que analizo más adelante.

Esta capacidad corresponde al **FastText propio con su tabla de subpalabras**. El archivo SUC `.vec` cargado como `KeyedVectors` contiene vectores de palabras completas; ese formato no conserva la información necesaria para generar nuevos vectores por subpalabras.

## 4.6 Selección del modelo propio para las siguientes secciones

Después de revisar las seis palabras de control, **elegí el Word2Vec base de noticias**: skip-gram, 100 dimensiones, ventana 5, `min_count=2`, `negative=5` y 10 épocas.

Mi elección es cualitativa: encontré grupos reconocibles para `banco` (`central`, `popular`, `república`, `agrario`, `davivienda`), `gobierno` (`congreso`, `electo`, `consejo`, `gremial`, `mandatario`) y `empresa` (`startup`, `compañía`, `emergente`). En `tecnología` obtuve `computación`, `cuántica` y `tecnológica`. También identifiqué vecinos menos claros, como `hueco` para `mercado` y `utilizará` para `tecnología`, por lo que no considero que todos sus resultados sean correctos.

La variante de 50 dimensiones también resultó competitiva y mejoró algunos ejemplos, como los vecinos financieros de `mercado`. Conservé la base por su conjunto interpretable de vecinos, particularmente para `gobierno`, y por mantener una dimensión intermedia con un costo de entrenamiento moderado. No encontré una mejora consistente que justificara usar 300 dimensiones o ventana 10. Esta revisión no demuestra que la base sea superior en todas las tareas; contrastaré la elección con las evaluaciones de la sección 5.

No utilicé el promedio de cosenos para elegir: un coseno más alto dentro de otro modelo no garantiza mejor calidad. En la sección 5 compararé este Word2Vec con el del Quijote, SVD y SUC.


In [69]:
MEJOR_VARIANTE = "base"  # Selección confirmada tras revisar las seis palabras de control.
if MEJOR_VARIANTE not in modelos_w2v_noticias:
    raise ValueError(f"Modelo desconocido. Opciones: {list(modelos_w2v_noticias)}")
mejor_w2v_noticias = modelos_w2v_noticias[MEJOR_VARIANTE]
print(f"Selección confirmada: {MEJOR_VARIANTE}")
display(tabla_entrenamiento_w2v[
    tabla_entrenamiento_w2v["Modelo"] == MEJOR_VARIANTE
].round(3))
with pd.option_context("display.max_colwidth", None):
    display(vecinos_w2v_noticias[
        vecinos_w2v_noticias["Modelo"] == MEJOR_VARIANTE
    ])

Selección confirmada: base


,Modelo,vector_size,window,min_count,sg,negative,sample,epochs,Palabras,Vocabulario (s),Entrenamiento (s),Total (s)
0,base,100,5,2,1,5,0.001,10,17129,0.169,10.315,10.483


,Modelo,Palabra de control,Vecino 1,Vecino 2,Vecino 3,Vecino 4,Vecino 5
0,base,banco,central (0.7007),popular (0.6011),república (0.5931),agrario (0.5663),davivienda (0.5465)
1,base,gobierno,congreso (0.7079),electo (0.6283),consejo (0.6189),gremial (0.6163),mandatario (0.6091)
2,base,empresa,startup (0.7208),compañía (0.6905),certificación (0.6560),emergente (0.6449),monitoreo (0.6421)
3,base,mercado,colombiano (0.5599),sector (0.5350),apetito (0.5199),accionario (0.5073),hueco (0.5048)
4,base,tecnología,estandarización (0.6801),computación (0.6659),cuántica (0.6564),tecnológica (0.6535),utilizará (0.6505)
5,base,sostenibilidad,digitalización (0.6481),diversidad (0.6374),alineado (0.6332),fundación (0.6222),prioridades (0.6134)


### Guardar los vectores y la configuración

Guardé los vectores del Word2Vec base seleccionado y del Quijote como `.kv`, con su configuración en un archivo JSON. Verifiqué que los archivos pudieran cargarse y conservaran sus vectores. Confirmé la selección de `base` tras revisar las salidas, por lo que los vectores guardados siguen correspondiendo al modelo elegido.


In [70]:
import json

RUTA_MEJOR_W2V = RAIZ / "Lab9/word2vec_noticias_mejor.kv"
RUTA_W2V_QUIJOTE = RAIZ / "Lab9/word2vec_quijote.kv"
mejor_w2v_noticias.wv.save(str(RUTA_MEJOR_W2V), separately=[])
modelo_w2v_quijote.wv.save(str(RUTA_W2V_QUIJOTE), separately=[])

# Las funciones no son serializables en JSON: registro su nombre.
def configuracion_serializable(configuracion):
    return {clave: (valor.__name__ if callable(valor) else valor)
            for clave, valor in configuracion.items()}

metadata_seccion4 = {
    "mejor_variante": MEJOR_VARIANTE,
    "seleccion_provisional": False,
    "configuracion_noticias": configuracion_serializable(CONFIGURACIONES_W2V[MEJOR_VARIANTE]),
    "configuracion_quijote": configuracion_serializable(CONFIG_BASE_W2V),
    "configuracion_fasttext": {
        **configuracion_serializable(CONFIG_BASE_W2V),
        "min_n": 3, "max_n": 6, "bucket": 200_000,
    },
    "palabras_control": PALABRAS_CONTROL,
    "palabras_comparacion_dominios": PALABRAS_DOMINIO,
    "consultas_oov": consultas_oov,
}
(RAIZ / "Lab9/configuracion_embeddings.json").write_text(
    json.dumps(metadata_seccion4, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
for ruta, modelo in [(RUTA_MEJOR_W2V, mejor_w2v_noticias),
                     (RUTA_W2V_QUIJOTE, modelo_w2v_quijote)]:
    recargado = KeyedVectors.load(str(ruta))
    assert recargado.index_to_key == modelo.wv.index_to_key
    np.testing.assert_allclose(recargado.vectors, modelo.wv.vectors)
    print(f"Guardado y verificado: {ruta.relative_to(RAIZ)} · "
          f"{len(recargado):,} palabras × {recargado.vector_size} dimensiones")

Guardado y verificado: Lab9/word2vec_noticias_mejor.kv · 17,129 palabras × 100 dimensiones
Guardado y verificado: Lab9/word2vec_quijote.kv · 11,760 palabras × 100 dimensiones


## 4.7 Análisis de mis resultados

### Configuraciones y costo de entrenamiento

Entrené seis modelos Word2Vec sobre noticias con el mismo vocabulario de **17,129 palabras**. Mantuve constantes `min_count=2`, `negative=5`, `sample=0.001` y **10 épocas**; en cada variante cambié solo un parámetro respecto de la base.

| Modelo | Arquitectura | Ventana | Dimensiones | Entrenamiento (s) | Total con vocabulario (s) |
|---|---|---:|---:|---:|---:|
| Base | Skip-gram | 5 | 100 | 10.315 | 10.483 |
| CBOW | CBOW | 5 | 100 | 3.161 | 3.287 |
| Ventana 2 | Skip-gram | 2 | 100 | 5.754 | 5.874 |
| Ventana 10 | Skip-gram | 10 | 100 | 17.321 | 17.438 |
| Dimensión 50 | Skip-gram | 5 | 50 | 8.614 | 8.733 |
| Dimensión 300 | Skip-gram | 5 | 300 | 20.379 | 20.511 |

Encontré que CBOW fue el más rápido: tardó aproximadamente **3.26 veces menos** que la base. Aumentar la ventana o la dimensión incrementó el tiempo en esta ejecución. Estos tiempos corresponden a mi entorno y a una sola corrida; no son una comparación general de rendimiento entre equipos.

### CBOW frente a skip-gram

Con la base obtuve para `banco` vecinos como `central`, `popular`, `república`, `agrario` y `davivienda`, que pude relacionar con el ámbito bancario. En CBOW aparecieron `cordillera`, `grupo`, `gradualidad`, `equipo` y `área`, un conjunto menos claro para la consulta financiera.

CBOW sí produjo asociaciones interpretables en otras consultas: para `tecnología` encontré `digitalización`, `tecnológica`, `transformación` y `blockchain`; para `sostenibilidad`, `innovación` y `descarbonización`. Por ello, no concluí que CBOW fuera siempre peor. En mi revisión, la base ofreció una combinación útil de vecinos bancarios, políticos y empresariales, mientras CBOW destacó por su menor tiempo.

### Ventanas pequeñas y grandes

No encontré una separación consistente entre vecinos sintácticos con ventana 2 y vecinos temáticos con ventana 10. Ambas ventanas devolvieron sobre todo relaciones temáticas, nombres y asociaciones de uso, sin que la pequeña produjera principalmente variantes de la misma palabra.

Con ventana 2, `empresa` se acercó a `startup` y `compañía`, pero también a `jurisdicción` y `película`. Con ventana 10 aparecieron `compañía`, `enfocada` y `startup`, junto con nombres específicos como `consentio` y `vincent`. Para `mercado`, la ventana 10 incorporó `accionario` y `bursátil`, mientras la 2 incluyó `sector`, `panorama` y `fútbol`.

Estos ejemplos me permitieron observar un cambio de contexto, pero no confirmar de forma general la tendencia esperada. La ventana 10 tardó **17.321 s**, frente a **5.754 s** de la ventana 2, sin mejorar todas las consultas.

### Dimensiones 50 y 300

La variante de **50 dimensiones** produjo vecinos financieros reconocibles para `banco` y `mercado`: `central`, `agrario`, `popular`, `entidad`, `segmento` y `solvencia`, según la consulta. Para `tecnología` obtuve `innovación`, `estandarización`, `metodologías`, `computación` y `cuántica`.

Al aumentar a **300 dimensiones**, no observé una mejora uniforme. Para `empresa` aparecieron `startup`, `compañía`, `emergente` y `participada`, pero para `mercado` también encontré términos poco específicos como `dificultan` y `tradicionalmente`. El entrenamiento tardó **20.379 s**, frente a **8.614 s** con 50 dimensiones. Con estas salidas no encontré una razón suficiente para preferir el modelo más grande.

### El significado viene del corpus

El modelo del Quijote tuvo **11,760 palabras** y tardó **6.185 s** en entrenarse. Comparé cinco palabras compartidas y encontré estos contrastes:

| Palabra | Ejemplos de noticias | Ejemplos del Quijote | Lo que observé |
|---|---|---|---|
| señor | encuadernador, sustentabilidad, unimos | hermano, álvaro, castellano, tarfe | En el Quijote aparecieron tratamientos y nombres; en noticias, el conjunto fue poco coherente. |
| dinero | vivir, estorbo, crediticio | arráez, hallazgo, dueño | Los vecinos cambiaron y no formaron un grupo financiero claro en ambos modelos. |
| casa | pasiva, teletrabajo, filtro | aldea, corte, hacienda | Encontré asociaciones modernas en noticias y espacios o posesiones en la novela. |
| palabra | sucedería, inevitable, viable | hablar, respondelle, hablase, responder | En el Quijote identifiqué una relación clara con el habla y el diálogo. |
| trabajo | perfil, remoto, colaborativo, teletrabajo | recogimiento, descanso, alivio, daño | Noticias reflejó empleo contemporáneo; la novela, contextos ligados al descanso o la dificultad. |

Entendí que una palabra no recibe un significado fijo independiente del texto: sus vecinos dependen de sus contextos de entrenamiento. También observé resultados débiles, especialmente para `señor` y `palabra` en noticias. No atribuí todas las diferencias únicamente al dominio, porque los corpus tienen tamaños y frecuencias diferentes y algunos vectores pueden estar peor estimados.

### Palabras desconocidas con FastText

Entrené FastText con **17,129 palabras y 100 dimensiones**; tardó **23.133 s**. Comprobé que los tres ejemplos tenían **cero apariciones** en noticias y no estaban registrados como palabras completas en su vocabulario.

| Caso | Consulta | Vecinos de FastText | Resultado con Word2Vec |
|---|---|---|---|
| Error de dedo | gobiernno | gobierno (0.9724), gobiernos (0.8827), niinisto (0.7460), gobernar (0.7146), gobernador (0.7051) | `KeyError` |
| Conjugación poco común | gobernásemos | decidimos (0.8565), coincidimos (0.8559), reforzamos (0.8499), andemos (0.8486), anónimos (0.8465) | `KeyError` |
| Palabra inventada | criptoempresa | empresa (0.8388), criptomoneda (0.7808), criptografía (0.7746), cripto (0.7650), impresora (0.7622) | `KeyError` |

FastText generó vectores de **100 coordenadas** para los tres casos. Para `gobiernno`, la cercanía con `gobierno` mostró una respuesta útil ante el error de dedo. En `criptoempresa` encontré asociaciones con ambos componentes, aunque `impresora` fue menos evidente.

En cambio, `gobernásemos` se acercó a varias formas terminadas en `-mos` y a `anónimos`, sin recuperar claramente la relación semántica con gobernar. Esto me mostró que compartir fragmentos de caracteres ayuda a construir un vector, pero no garantiza comprender la palabra ni su significado. Word2Vec no pudo producir vectores para ninguno de los tres ejemplos porque depende de entradas de palabras completas.

### Modelo elegido y alcance de la ejecución

Confirmé el **modelo base de noticias** como mi elección para continuar, con la justificación cualitativa de la sección 4.6. Conservé `word2vec_noticias_mejor.kv` (**17,129 × 100**) y `word2vec_quijote.kv` (**11,760 × 100**). Al revisar ambos archivos comprobé que todos sus valores fueran finitos y que no hubiera vectores nulos.

Durante el entrenamiento aparecieron mensajes internos de gensim: `Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'`. La ejecución continuó y produjo las tablas y archivos, pero registré esos mensajes como una limitación del entorno. Las comprobaciones de los vectores guardados no demuestran por sí solas que no haya habido ningún efecto numérico; no atribuí a esos mensajes una causa concreta ni los eliminé de las salidas.


# 5. Similitud, vecinos y analogías

Para esta comparación elegí los cuatro modelos solicitados: **SVD sobre PPMI**, mi **Word2Vec de noticias seleccionado**, el **Word2Vec del Quijote** y **fastText SUC**. Definí las consultas y los puntajes de referencia antes de observar las salidas de esta sección.

Mantuve las minúsculas y las tildes de la sección 1. Registré explícitamente las palabras fuera del vocabulario: no las reemplacé por cero ni por otra grafía. Utilicé mis funciones para los cálculos y gensim como referencia de verificación.

### Cargar los cuatro modelos

Puedo ejecutar esta sección desde los archivos guardados, sin repetir el entrenamiento de la sección 4. Si SUC ya está cargado con la misma ruta y límite, reutilizaré ese objeto. En otro equipo tendré que recuperar los archivos `.kv` y el modelo SUC, que están excluidos de Git.

In [71]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from gensim.models import KeyedVectors
from IPython.display import display

if "RAIZ" not in globals():
    RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "Lab9/lab9.ipynb").is_file()), None)
    if RAIZ is None:
        raise FileNotFoundError("Abre el notebook dentro del repositorio NLP.")

rutas_modelos_propios = {
    "SVD-PPMI": RAIZ / "Lab9/ppmi_svd_100.kv",
    "Word2Vec noticias": RAIZ / "Lab9/word2vec_noticias_mejor.kv",
    "Word2Vec Quijote": RAIZ / "Lab9/word2vec_quijote.kv",
}
faltan_archivos = [str(p) for p in rutas_modelos_propios.values() if not p.is_file()]
if faltan_archivos:
    raise FileNotFoundError(f"Recupera o genera estos vectores antes de continuar: {faltan_archivos}")
modelos_seccion5 = {nombre: KeyedVectors.load(str(ruta))
                    for nombre, ruta in rutas_modelos_propios.items()}

RUTA_SUC = Path(globals().get("RUTA_SUC", RAIZ / "Lab9/embeddings-m-model.vec"))
LIMITE_SUC = globals().get("LIMITE_SUC", 200_000)
firma_suc = (str(RUTA_SUC.resolve()), LIMITE_SUC)
if "suc" not in globals() or globals().get("_firma_suc_cargada") != firma_suc:
    if not RUTA_SUC.is_file():
        raise FileNotFoundError(f"No encontré SUC en {RUTA_SUC}.")
    suc = KeyedVectors.load_word2vec_format(str(RUTA_SUC), limit=LIMITE_SUC)
    _firma_suc_cargada = firma_suc
modelos_seccion5["fastText SUC"] = suc

# Conservo una matriz normalizada por modelo para evitar repetir este cálculo.
vectores_unitarios_5 = {}
for nombre, modelo in modelos_seccion5.items():
    modelo.fill_norms()
    if not np.isfinite(modelo.vectors).all() or np.any(modelo.norms == 0):
        raise ValueError(f"Revisa {nombre}: encontré vectores no finitos o de norma cero.")
    vectores_unitarios_5[nombre] = modelo.get_normed_vectors()

display(pd.DataFrame([
    {"Modelo": nombre, "Palabras": len(modelo), "Dimensiones": modelo.vector_size}
    for nombre, modelo in modelos_seccion5.items()
]))

,Modelo,Palabras,Dimensiones
0,SVD-PPMI,2000,100
1,Word2Vec noticias,17129,100
2,Word2Vec Quijote,11760,100
3,fastText SUC,200000,100


## 5.1 Similitud coseno y vecinos

### Función propia y verificación

Implementé el coseno con NumPy. Verifiqué tres pares presentes en los cuatro modelos y obtuve doce comprobaciones contra `similarity` de gensim. La selección de respaldo depende solo de la cobertura y no de los valores de similitud.

In [72]:
def coseno(u, v):
    u = np.asarray(u, dtype=np.float64)
    v = np.asarray(v, dtype=np.float64)
    denominador = np.linalg.norm(u) * np.linalg.norm(v)
    if denominador == 0:
        raise ValueError("El coseno no está definido para vectores de norma cero.")
    return float(np.dot(u, v) / denominador)


vocabulario_comun_5 = set.intersection(
    *(set(modelo.key_to_index) for modelo in modelos_seccion5.values())
)
candidatos_verificacion = [
    ("empresa", "compañía"), ("gobierno", "estado"), ("casa", "dinero"),
    ("hombre", "mujer"), ("trabajo", "tiempo"), ("vida", "mundo"),
]
pares_verificacion = [(a, b) for a, b in candidatos_verificacion
                      if a in vocabulario_comun_5 and b in vocabulario_comun_5][:3]
if len(pares_verificacion) < 3:
    raise ValueError("Selecciona tres pares con cobertura en los cuatro modelos.")
filas_verificacion = []
for nombre, modelo in modelos_seccion5.items():
    for a, b in pares_verificacion:
        propio = coseno(modelo[a], modelo[b])
        referencia = float(modelo.similarity(a, b))
        coincide = bool(np.isclose(propio, referencia, atol=1e-6, rtol=1e-5))
        filas_verificacion.append({"Modelo": nombre, "Par": f"{a} / {b}",
            "Coseno propio": propio, "gensim": referencia,
            "Diferencia absoluta": abs(propio - referencia), "Coincide": coincide})
verificacion_coseno = pd.DataFrame(filas_verificacion)
assert verificacion_coseno["Coincide"].all()
display(verificacion_coseno.round(8))

,Modelo,Par,Coseno propio,gensim,Diferencia absoluta,Coincide
0,SVD-PPMI,empresa / compañía,0.704364,0.704364,4.000000e-08,True
1,SVD-PPMI,gobierno / estado,0.467866,0.467866,2.000000e-08,True
2,SVD-PPMI,casa / dinero,0.330934,0.330934,0.000000e+00,True
3,Word2Vec noticias,empresa / compañía,0.690544,0.690544,3.000000e-08,True
4,Word2Vec noticias,gobierno / estado,0.560520,0.560521,4.000000e-08,True
5,Word2Vec noticias,casa / dinero,0.331348,0.331348,3.000000e-08,True
6,Word2Vec Quijote,empresa / compañía,0.637212,0.637212,4.000000e-08,True
7,Word2Vec Quijote,gobierno / estado,0.673381,0.673381,1.000000e-08,True
8,Word2Vec Quijote,casa / dinero,0.487759,0.487759,0.000000e+00,True
9,fastText SUC,empresa / compañía,0.823897,0.823897,1.000000e-08,True


### Vecinos de cinco palabras compartidas

Elegí cinco palabras con cobertura en los cuatro modelos para comparar cinco vecinas por consulta. No restringí los vecinos a un vocabulario común: cada modelo buscó dentro de su propio espacio. Esa diferencia de candidatos también influye en las listas.

In [73]:
candidatas_vecinos_5 = [
    "banco", "gobierno", "empresa", "casa", "dinero", "palabra", "trabajo", "hombre", "mujer",
]
PALABRAS_VECINOS_5 = [p for p in candidatas_vecinos_5 if p in vocabulario_comun_5][:5]
if len(PALABRAS_VECINOS_5) != 5:
    raise ValueError("Selecciona cinco palabras presentes en los cuatro modelos.")
print("Consultas comunes:", PALABRAS_VECINOS_5)
filas_vecinos_5 = []
for palabra in PALABRAS_VECINOS_5:
    fila = {"Palabra": palabra}
    for nombre, modelo in modelos_seccion5.items():
        fila[nombre] = "; ".join(f"{w} ({s:.4f})"
                                 for w, s in modelo.most_similar(palabra, topn=5))
    filas_vecinos_5.append(fila)
vecinos_cuatro_modelos = pd.DataFrame(filas_vecinos_5)
with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(vecinos_cuatro_modelos)

Consultas comunes: ['banco', 'gobierno', 'empresa', 'casa', 'dinero']


,Palabra,SVD-PPMI,Word2Vec noticias,Word2Vec Quijote,fastText SUC
0,banco,bbva (0.6447); central (0.6343); grupo (0.5880); también (0.5855); colombia (0.5541),central (0.7007); popular (0.6011); república (0.5931); agrario (0.5663); davivienda (0.5465),encajó (0.9313); ristre (0.9306); correa (0.9278); bordón (0.9268); acelerada (0.9259),hipotecario (0.7651); bancos (0.7547); bank (0.7397); citibank (0.7385); monetario (0.7268)
1,gobierno,congreso (0.5833); banco (0.5389); país (0.5284); federal (0.5269); también (0.5206),congreso (0.7079); electo (0.6283); consejo (0.6189); gremial (0.6163); mandatario (0.6091),condado (0.7955); cobrar (0.7722); ahorrar (0.7681); regalo (0.7674); término (0.7657),elgobierno (0.7517); gobier (0.7453); go­bierno (0.7342); ministro (0.7250); gobiernos (0.7169)
2,empresa,compañía (0.7044); plataforma (0.6960); tecnología (0.6543); asociación (0.6321); entidad (0.6185),startup (0.7208); compañía (0.6905); certificación (0.6560); emergente (0.6449); monitoreo (0.6421),herida (0.8558); demanda (0.8552); novedad (0.8430); peligrosa (0.8382); fantasma (0.8197),compañía (0.8239); constructora (0.7936); subsidiaria (0.7881); comercializadora (0.7869); aseguradora (0.7833)
3,casa,coche (0.5419); juntos (0.5098); fuerza (0.4869); vida (0.4798); área (0.4727),pasiva (0.6573); caminar (0.6197); teletrabajo (0.6158); filtro (0.6133); compatibilizar (0.6046),aldea (0.6878); corte (0.6128); mujer (0.5987); hacienda (0.5966); majestad (0.5959),mansión (0.7963); casita (0.7862); cabaña (0.7546); choza (0.7536); habitación (0.7362)
4,dinero,información (0.5225); manejo (0.5125); préstamo (0.5059); celular (0.5019); tiempo (0.4968),vivir (0.7080); estorbo (0.6843); crediticio (0.6823); virtuoso (0.6662); click (0.6630),arráez (0.8421); hallazgo (0.8356); dueño (0.8353); tema (0.8317); haréis (0.8266),dinerito (0.8059); dineral (0.7802); centavo (0.7767); gastarlo (0.7630); dinerillo (0.7511)


### Antónimos: similitud distribucional y sinonimia

Elegí cuatro pares de antónimos. Interpreté los cosenos altos como semejanza entre contextos de uso, no como equivalencia de significado: palabras opuestas pueden ocupar posiciones y tratar temas similares. Detallé los valores concretos en la sección 5.4.

In [74]:
PARES_ANTONIMOS = [("bueno", "malo"), ("subir", "bajar"),
                   ("ganar", "perder"), ("público", "privado")]
filas_antonimos = []
for nombre, modelo in modelos_seccion5.items():
    for a, b in PARES_ANTONIMOS:
        faltantes = [p for p in (a, b) if p not in modelo.key_to_index]
        filas_antonimos.append({
            "Modelo": nombre, "Par": f"{a} / {b}",
            "Coseno": coseno(modelo[a], modelo[b]) if not faltantes else np.nan,
            "Fuera del vocabulario": ", ".join(faltantes) if faltantes else "Ninguna",
        })
similitudes_antonimos = pd.DataFrame(filas_antonimos)
display(similitudes_antonimos.round({"Coseno": 6}))

,Modelo,Par,Coseno,Fuera del vocabulario
0,SVD-PPMI,bueno / malo,NaN,"bueno, malo"
1,SVD-PPMI,subir / bajar,NaN,bajar
2,SVD-PPMI,ganar / perder,NaN,"ganar, perder"
3,SVD-PPMI,público / privado,0.553847,Ninguna
4,Word2Vec noticias,bueno / malo,0.702509,Ninguna
5,Word2Vec noticias,subir / bajar,0.744858,Ninguna
6,Word2Vec noticias,ganar / perder,0.645275,Ninguna
7,Word2Vec noticias,público / privado,0.657548,Ninguna
8,Word2Vec Quijote,bueno / malo,0.799807,Ninguna
9,Word2Vec Quijote,subir / bajar,0.766885,Ninguna


## 5.2 Mi conjunto de similitud

Definí **20 pares** con puntajes de **0 a 10** como referencia subjetiva: asigné valores altos a significados muy parecidos, intermedios a relaciones temáticas y bajos a términos sin relación evidente. Separé esta referencia de las predicciones del modelo; estos puntajes se pueden revisar antes de ejecutar la comparación si se desea expresar un juicio distinto.

Calculé Spearman sobre los pares cubiertos por cada modelo y reporté los excluidos. Además, repetí el cálculo sobre los pares cubiertos **por los cuatro modelos** para comparar el mismo conjunto. Dejé una comprobación que devuelve `NaN` si hay menos de tres pares o valores constantes, evitando interpretar una correlación indefinida. Utilicé el coeficiente de [Spearman de SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.spearmanr.html) como descripción exploratoria de este conjunto pequeño.

In [75]:
gold_similitud = pd.DataFrame([
    ("empresa", "compañía", 10, "Muy parecidas"),
    ("casa", "hogar", 9, "Muy parecidas"),
    ("dinero", "moneda", 8, "Muy parecidas"),
    ("trabajo", "empleo", 9, "Muy parecidas"),
    ("gobierno", "estado", 8, "Muy parecidas"),
    ("banco", "entidad", 6, "Relacionadas"),
    ("rey", "reina", 7, "Relacionadas"),
    ("hombre", "mujer", 5, "Relacionadas"),
    ("caballo", "animal", 7, "Relacionadas"),
    ("banco", "dinero", 6, "Relacionadas"),
    ("gobierno", "ley", 6, "Relacionadas"),
    ("mercado", "economía", 7, "Relacionadas"),
    ("tecnología", "innovación", 7, "Relacionadas"),
    ("empresa", "mercado", 5, "Relacionadas"),
    ("casa", "familia", 5, "Relacionadas"),
    ("empresa", "caballo", 0, "Sin relación directa"),
    ("gobierno", "zapato", 0, "Sin relación directa"),
    ("dinero", "luna", 1, "Sin relación directa"),
    ("banco", "poesía", 1, "Sin relación directa"),
    ("tecnología", "pan", 1, "Sin relación directa"),
], columns=["Palabra A", "Palabra B", "Puntaje personal", "Grupo"])
gold_similitud.index = pd.RangeIndex(1, 21, name="Par")
display(gold_similitud)

filas_gold = []
for id_par, fila in gold_similitud.iterrows():
    a, b = fila["Palabra A"], fila["Palabra B"]
    for nombre, modelo in modelos_seccion5.items():
        faltantes = [p for p in (a, b) if p not in modelo.key_to_index]
        filas_gold.append({
            "Par": id_par, "Modelo": nombre, "Palabra A": a, "Palabra B": b,
            "Puntaje personal": fila["Puntaje personal"],
            "Coseno": coseno(modelo[a], modelo[b]) if not faltantes else np.nan,
            "Cubierto": not faltantes, "Palabras OOV": ", ".join(faltantes),
        })
resultados_gold = pd.DataFrame(filas_gold)


def spearman_seguro(datos):
    if len(datos) < 3 or datos["Puntaje personal"].nunique() < 2 or datos["Coseno"].nunique() < 2:
        return np.nan
    return float(spearmanr(datos["Puntaje personal"], datos["Coseno"]).statistic)


pares_gold_comunes = resultados_gold.groupby("Par")["Cubierto"].all()
ids_gold_comunes = pares_gold_comunes[pares_gold_comunes].index
filas_spearman = []
for nombre, grupo in resultados_gold.groupby("Modelo", sort=False):
    cubiertos = grupo[grupo["Cubierto"]]
    comunes = grupo[grupo["Par"].isin(ids_gold_comunes)]
    filas_spearman.append({
        "Modelo": nombre, "Pares evaluados": len(cubiertos),
        "Pares OOV": int((~grupo["Cubierto"]).sum()),
        "Spearman (cobertura propia)": spearman_seguro(cubiertos),
        "Pares comunes": len(comunes),
        "Spearman (pares comunes)": spearman_seguro(comunes),
    })
resumen_spearman = pd.DataFrame(filas_spearman)
display(resultados_gold.pivot(index="Par", columns="Modelo", values="Coseno").join(
    gold_similitud[["Palabra A", "Palabra B", "Puntaje personal"]]
).round(5))
display(resumen_spearman.round(5))
print("Pares con cobertura en todos los modelos:", list(ids_gold_comunes))
print("Pares excluidos por modelo:")
display(resultados_gold.loc[~resultados_gold["Cubierto"],
    ["Modelo", "Par", "Palabra A", "Palabra B", "Palabras OOV"]])

,Palabra A,Palabra B,Puntaje personal,Grupo
Par,,,,
1,empresa,compañía,10,Muy parecidas
2,casa,hogar,9,Muy parecidas
3,dinero,moneda,8,Muy parecidas
4,trabajo,empleo,9,Muy parecidas
5,gobierno,estado,8,Muy parecidas
6,banco,entidad,6,Relacionadas
7,rey,reina,7,Relacionadas
8,hombre,mujer,5,Relacionadas
9,caballo,animal,7,Relacionadas


,SVD-PPMI,Word2Vec Quijote,Word2Vec noticias,fastText SUC,Palabra A,Palabra B,Puntaje personal
Par,,,,,,,
1,0.70436,0.63721,0.69054,0.82390,empresa,compañía,10
2,0.38544,NaN,0.37043,0.60759,casa,hogar,9
3,0.17317,0.52281,0.30540,0.50450,dinero,moneda,8
4,0.30019,NaN,0.43951,0.69984,trabajo,empleo,9
5,0.46787,0.67338,0.56052,0.56470,gobierno,estado,8
6,0.49577,NaN,0.51440,0.41928,banco,entidad,6
7,NaN,0.55590,0.77721,0.76117,rey,reina,7
8,NaN,0.44644,0.64775,0.58740,hombre,mujer,5
9,NaN,0.42749,NaN,0.60471,caballo,animal,7


,Modelo,Pares evaluados,Pares OOV,Spearman (cobertura propia),Pares comunes,Spearman (pares comunes)
0,SVD-PPMI,12,8,0.20215,6,0.35309
1,Word2Vec noticias,16,4,0.39705,6,0.55907
2,Word2Vec Quijote,13,7,0.36289,6,0.38252
3,fastText SUC,20,0,0.72641,6,0.26482


Pares con cobertura en todos los modelos: [1, 3, 5, 10, 11, 15]
Pares excluidos por modelo:


,Modelo,Par,Palabra A,Palabra B,Palabras OOV
6,Word2Vec Quijote,2,casa,hogar,hogar
14,Word2Vec Quijote,4,trabajo,empleo,empleo
22,Word2Vec Quijote,6,banco,entidad,entidad
24,SVD-PPMI,7,rey,reina,"rey, reina"
28,SVD-PPMI,8,hombre,mujer,"hombre, mujer"
32,SVD-PPMI,9,caballo,animal,"caballo, animal"
33,Word2Vec noticias,9,caballo,animal,caballo
46,Word2Vec Quijote,12,mercado,economía,"mercado, economía"
50,Word2Vec Quijote,13,tecnología,innovación,"tecnología, innovación"
54,Word2Vec Quijote,14,empresa,mercado,mercado


## 5.3 Analogías

### Aritmética vectorial y función propia

Para resolver «$a$ es a $b$ como $c$ es a ?», implementé la dirección $b-a+c$ y ordené las palabras candidatas por coseno, excluyendo las tres palabras de consulta.

**Normalización y gensim.** `most_similar(positive=[b,c], negative=[a])` normaliza cada vector de entrada antes de combinarlos. Para verificar la coincidencia utilicé $\hat b-\hat a+\hat c$ y normalicé el resultado. La división por tres que usa gensim no cambia su dirección. También dejé el modo `normalizar_entradas=False` para calcular $b-a+c$ con los vectores originales y mostrar que ambos procedimientos pueden producir vecinos distintos. Consulté la documentación de [KeyedVectors](https://radimrehurek.com/gensim/models/keyedvectors.html).

Fijé **18 analogías de seis tipos**, sin modificarlas en función de los aciertos. Para cada una reporté cinco candidatos y la posición de la respuesta correcta en todo el vocabulario disponible. Cuando la respuesta no quedó entre los cinco primeros pero sí estaba en el vocabulario, conservé su posición.

In [76]:
analogias_propias = pd.DataFrame([
    ("Género y parentesco", "hombre", "rey", "mujer", "reina"),
    ("Género y parentesco", "hombre", "padre", "mujer", "madre"),
    ("Género y parentesco", "hombre", "hermano", "mujer", "hermana"),
    ("País-capital", "españa", "madrid", "francia", "parís"),
    ("País-capital", "colombia", "bogotá", "perú", "lima"),
    ("País-capital", "italia", "roma", "alemania", "berlín"),
    ("País-gentilicio", "españa", "español", "francia", "francés"),
    ("País-gentilicio", "colombia", "colombiano", "méxico", "mexicano"),
    ("País-gentilicio", "perú", "peruano", "chile", "chileno"),
    ("Singular-plural", "empresa", "empresas", "banco", "bancos"),
    ("Singular-plural", "gobierno", "gobiernos", "mercado", "mercados"),
    ("Singular-plural", "casa", "casas", "palabra", "palabras"),
    ("Tiempo verbal", "comía", "comerá", "vivía", "vivirá"),
    ("Tiempo verbal", "trabajaba", "trabajará", "estudiaba", "estudiará"),
    ("Tiempo verbal", "ganaba", "ganará", "perdía", "perderá"),
    ("Adjetivo masculino-femenino", "bueno", "buena", "malo", "mala"),
    ("Adjetivo masculino-femenino", "público", "pública", "privado", "privada"),
    ("Adjetivo masculino-femenino", "alto", "alta", "bajo", "baja"),
], columns=["Tipo", "a", "b", "c", "Respuesta"])
analogias_propias.index = pd.RangeIndex(1, len(analogias_propias) + 1, name="Analogía")
display(analogias_propias)


def ordenar_analogia(a, b, c, nombre_modelo, excluir=True, normalizar_entradas=True):
    modelo = modelos_seccion5[nombre_modelo]
    faltantes = [p for p in (a, b, c) if p not in modelo.key_to_index]
    if faltantes:
        raise KeyError(f"Palabras de consulta fuera del vocabulario: {faltantes}")
    a_vec = modelo.get_vector(a, norm=normalizar_entradas)
    b_vec = modelo.get_vector(b, norm=normalizar_entradas)
    c_vec = modelo.get_vector(c, norm=normalizar_entradas)
    direccion = (b_vec + c_vec - a_vec) / 3.0
    norma = np.linalg.norm(direccion)
    if norma == 0 or not np.isfinite(norma):
        raise ValueError("La dirección de la analogía no es válida.")
    direccion = direccion / norma
    puntuaciones = vectores_unitarios_5[nombre_modelo] @ direccion
    orden = np.argsort(-puntuaciones, kind="stable")
    if excluir:
        prohibidos = {modelo.key_to_index[p] for p in (a, b, c)}
        orden = np.array([i for i in orden if i not in prohibidos], dtype=np.int64)
    return orden, puntuaciones


def analogia(a, b, c, k=5, nombre_modelo="Word2Vec noticias",
             excluir=True, normalizar_entradas=True):
    if k < 1:
        raise ValueError("k debe ser positivo.")
    orden, puntuaciones = ordenar_analogia(
        a, b, c, nombre_modelo, excluir=excluir,
        normalizar_entradas=normalizar_entradas,
    )
    modelo = modelos_seccion5[nombre_modelo]
    return [(modelo.index_to_key[i], float(puntuaciones[i])) for i in orden[:k]]


# Tres consultas compartidas: verifico la implementación, no sus aciertos semánticos.
palabras_verificacion_analogia = [p for p in
    ["hombre", "mujer", "casa", "dinero", "trabajo", "gobierno", "empresa", "palabra"]
    if p in vocabulario_comun_5]
if len(palabras_verificacion_analogia) < 5:
    raise ValueError("Necesito cinco palabras comunes para verificar las analogías.")
consultas_verificacion_analogia = [
    tuple(palabras_verificacion_analogia[i:i+3]) for i in range(3)
]
filas_verificacion_analogia = []
for nombre, modelo in modelos_seccion5.items():
    for a, b, c in consultas_verificacion_analogia:
        propio = analogia(a, b, c, nombre_modelo=nombre)
        referencia = modelo.most_similar(positive=[b, c], negative=[a], topn=5)
        mismas_palabras = [w for w, _ in propio] == [w for w, _ in referencia]
        mismos_cosenos = np.allclose([s for _, s in propio], [s for _, s in referencia],
                                    atol=1e-6, rtol=1e-5)
        filas_verificacion_analogia.append({
            "Modelo": nombre, "Consulta": f"{a} : {b} :: {c} : ?",
            "Mismo top 5": mismas_palabras, "Cosenos coinciden": bool(mismos_cosenos),
        })
verificacion_analogias = pd.DataFrame(filas_verificacion_analogia)
display(verificacion_analogias)
assert verificacion_analogias[["Mismo top 5", "Cosenos coinciden"]].to_numpy().all()

,Tipo,a,b,c,Respuesta
Analogía,,,,,
1,Género y parentesco,hombre,rey,mujer,reina
2,Género y parentesco,hombre,padre,mujer,madre
3,Género y parentesco,hombre,hermano,mujer,hermana
4,País-capital,españa,madrid,francia,parís
5,País-capital,colombia,bogotá,perú,lima
6,País-capital,italia,roma,alemania,berlín
7,País-gentilicio,españa,español,francia,francés
8,País-gentilicio,colombia,colombiano,méxico,mexicano
9,País-gentilicio,perú,peruano,chile,chileno


,Modelo,Consulta,Mismo top 5,Cosenos coinciden
0,SVD-PPMI,casa : dinero :: trabajo : ?,True,True
1,SVD-PPMI,dinero : trabajo :: gobierno : ?,True,True
2,SVD-PPMI,trabajo : gobierno :: empresa : ?,True,True
3,Word2Vec noticias,casa : dinero :: trabajo : ?,True,True
4,Word2Vec noticias,dinero : trabajo :: gobierno : ?,True,True
5,Word2Vec noticias,trabajo : gobierno :: empresa : ?,True,True
6,Word2Vec Quijote,casa : dinero :: trabajo : ?,True,True
7,Word2Vec Quijote,dinero : trabajo :: gobierno : ?,True,True
8,Word2Vec Quijote,trabajo : gobierno :: empresa : ?,True,True
9,fastText SUC,casa : dinero :: trabajo : ?,True,True


### Resultados por analogía y por tipo

Consideré evaluable una analogía cuando sus tres palabras de consulta **y su respuesta** estén en el vocabulario del modelo. Cuando solo faltó la respuesta, mostré los candidatos y marqué la analogía como no evaluable.

Reporté top 1 y top 5 sobre las analogías evaluables de cada tipo y, por separado, sobre todas las propuestas, contando los casos OOV como fallos de cobertura. Añadí un resumen sobre las analogías compartidas por los cuatro modelos. Así distinguí la capacidad de resolver una relación de la disponibilidad de sus palabras.

In [77]:
filas_analogias = []
for id_analogia, fila in analogias_propias.iterrows():
    a, b, c, respuesta = (fila[col] for col in ["a", "b", "c", "Respuesta"])
    for nombre, modelo in modelos_seccion5.items():
        faltantes = list(dict.fromkeys(p for p in (a, b, c, respuesta)
                                      if p not in modelo.key_to_index))
        consulta_cubierta = all(p in modelo.key_to_index for p in (a, b, c))
        top = []
        posicion = None
        if consulta_cubierta:
            orden, puntuaciones = ordenar_analogia(a, b, c, nombre)
            top = [(modelo.index_to_key[i], float(puntuaciones[i])) for i in orden[:5]]
            if respuesta in modelo.key_to_index:
                posiciones = np.flatnonzero(orden == modelo.key_to_index[respuesta])
                posicion = int(posiciones[0]) + 1 if len(posiciones) else None
        filas_analogias.append({
            "Analogía": id_analogia, "Tipo": fila["Tipo"], "Modelo": nombre,
            "Consulta": f"{a} : {b} :: {c} : ?", "Respuesta": respuesta,
            "Top 5": "; ".join(f"{w} ({s:.4f})" for w, s in top) if top else "Consulta OOV",
            "Posición correcta": posicion, "Evaluable": not faltantes,
            "Palabras OOV": ", ".join(faltantes),
            "Acierto top 1": posicion == 1,
            "Acierto top 5": posicion is not None and posicion <= 5,
        })
resultados_analogias = pd.DataFrame(filas_analogias)
resultados_analogias["Posición correcta"] = resultados_analogias["Posición correcta"].astype("Int64")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None,
                       "display.max_columns", None):
    display(resultados_analogias)


def resumir_analogias(grupo):
    total, evaluables = len(grupo), int(grupo["Evaluable"].sum())
    aciertos1, aciertos5 = int(grupo["Acierto top 1"].sum()), int(grupo["Acierto top 5"].sum())
    return {
        "Propuestas": total, "Evaluables": evaluables, "OOV": total - evaluables,
        "Top 1 evaluables (%)": 100 * aciertos1 / evaluables if evaluables else np.nan,
        "Top 5 evaluables (%)": 100 * aciertos5 / evaluables if evaluables else np.nan,
        "Top 1 todas (%)": 100 * aciertos1 / total if total else np.nan,
        "Top 5 todas (%)": 100 * aciertos5 / total if total else np.nan,
    }


resumen_analogias_tipo = pd.DataFrame([
    {"Modelo": nombre, "Tipo": tipo, **resumir_analogias(grupo)}
    for (nombre, tipo), grupo in resultados_analogias.groupby(["Modelo", "Tipo"], sort=False)
])
resumen_analogias_global = pd.DataFrame([
    {"Modelo": nombre, **resumir_analogias(grupo)}
    for nombre, grupo in resultados_analogias.groupby("Modelo", sort=False)
])
ids_analogias_comunes = resultados_analogias.groupby("Analogía")["Evaluable"].all()
ids_analogias_comunes = ids_analogias_comunes[ids_analogias_comunes].index
resumen_analogias_comunes = pd.DataFrame([
    {"Modelo": nombre, **resumir_analogias(grupo[grupo["Analogía"].isin(ids_analogias_comunes)])}
    for nombre, grupo in resultados_analogias.groupby("Modelo", sort=False)
])
print("Aciertos por tipo:")
display(resumen_analogias_tipo.round(2))
print("Resumen global:")
display(resumen_analogias_global.round(2))
print("Mismo conjunto de analogías para todos los modelos:", list(ids_analogias_comunes))
display(resumen_analogias_comunes.round(2))

,Analogía,Tipo,Modelo,Consulta,Respuesta,Top 5,Posición correcta,Evaluable,Palabras OOV,Acierto top 1,Acierto top 5
0,1,Género y parentesco,SVD-PPMI,hombre : rey :: mujer : ?,reina,Consulta OOV,<NA>,False,"hombre, rey, mujer, reina",False,False
1,1,Género y parentesco,Word2Vec noticias,hombre : rey :: mujer : ?,reina,coordinador (0.7518); andrea (0.7445); científica (0.7276); ana (0.7231); francisco (0.7077),5118,True,,False,False
2,1,Género y parentesco,Word2Vec Quijote,hombre : rey :: mujer : ?,reina,patria (0.6303); iglesia (0.6266); madre (0.6118); ana (0.6034); infanta (0.5925),19,True,,False,False
3,1,Género y parentesco,fastText SUC,hombre : rey :: mujer : ?,reina,reina (0.7628); princesa (0.6654); consorte (0.6648); sibila (0.6532); isabel (0.6500),1,True,,True,True
4,2,Género y parentesco,SVD-PPMI,hombre : padre :: mujer : ?,madre,Consulta OOV,<NA>,False,"hombre, padre, mujer, madre",False,False
5,2,Género y parentesco,Word2Vec noticias,hombre : padre :: mujer : ?,madre,diversidad (0.6539); hábitat (0.6193); bandera (0.6179); independencia (0.6083); misión (0.6077),3932,True,,False,False
6,2,Género y parentesco,Word2Vec Quijote,hombre : padre :: mujer : ?,madre,hijos (0.6241); ana (0.6207); félix (0.6128); marido (0.5983); hija (0.5963),8,True,,False,False
7,2,Género y parentesco,fastText SUC,hombre : padre :: mujer : ?,madre,madre (0.8002); hija (0.7936); hermana (0.7810); esposa (0.7798); sobrina (0.7527),1,True,,True,True
8,3,Género y parentesco,SVD-PPMI,hombre : hermano :: mujer : ?,hermana,Consulta OOV,<NA>,False,"hombre, hermano, mujer, hermana",False,False
9,3,Género y parentesco,Word2Vec noticias,hombre : hermano :: mujer : ?,hermana,Consulta OOV,<NA>,False,"hermano, hermana",False,False


Aciertos por tipo:


,Modelo,Tipo,Propuestas,Evaluables,OOV,Top 1 evaluables (%),Top 5 evaluables (%),Top 1 todas (%),Top 5 todas (%)
0,SVD-PPMI,Género y parentesco,3,0,3,NaN,NaN,0.00,0.00
1,Word2Vec noticias,Género y parentesco,3,2,1,0.00,0.00,0.00,0.00
2,Word2Vec Quijote,Género y parentesco,3,3,0,0.00,0.00,0.00,0.00
3,fastText SUC,Género y parentesco,3,3,0,100.00,100.00,100.00,100.00
4,SVD-PPMI,País-capital,3,0,3,NaN,NaN,0.00,0.00
5,Word2Vec noticias,País-capital,3,3,0,0.00,0.00,0.00,0.00
6,Word2Vec Quijote,País-capital,3,1,2,0.00,0.00,0.00,0.00
7,fastText SUC,País-capital,3,3,0,33.33,100.00,33.33,100.00
8,SVD-PPMI,País-gentilicio,3,0,3,NaN,NaN,0.00,0.00
9,Word2Vec noticias,País-gentilicio,3,3,0,33.33,33.33,33.33,33.33


Resumen global:


,Modelo,Propuestas,Evaluables,OOV,Top 1 evaluables (%),Top 5 evaluables (%),Top 1 todas (%),Top 5 todas (%)
0,SVD-PPMI,18,4,14,25.00,25.00,5.56,5.56
1,Word2Vec noticias,18,14,4,7.14,7.14,5.56,5.56
2,Word2Vec Quijote,18,9,9,11.11,11.11,5.56,5.56
3,fastText SUC,18,18,0,83.33,100.00,83.33,100.00


Mismo conjunto de analogías para todos los modelos: [10, 18]


,Modelo,Propuestas,Evaluables,OOV,Top 1 evaluables (%),Top 5 evaluables (%),Top 1 todas (%),Top 5 todas (%)
0,SVD-PPMI,2,2,0,0.0,0.0,0.0,0.0
1,Word2Vec noticias,2,2,0,0.0,0.0,0.0,0.0
2,Word2Vec Quijote,2,2,0,0.0,0.0,0.0,0.0
3,fastText SUC,2,2,0,100.0,100.0,100.0,100.0


### Vectores originales frente a entradas normalizadas

En estas consultas comparé la operación con entradas normalizadas —la utilizada para verificar gensim— y la operación directa sobre vectores originales. La normalización de las palabras candidatas se mantiene en ambos casos para calcular el coseno. No asumí que las dos operaciones tuvieran que devolver el mismo resultado.

In [78]:
filas_operacion = []
for nombre in modelos_seccion5:
    for a, b, c in consultas_verificacion_analogia:
        normalizada = analogia(a, b, c, nombre_modelo=nombre)
        original = analogia(a, b, c, nombre_modelo=nombre, normalizar_entradas=False)
        filas_operacion.append({
            "Modelo": nombre, "Consulta": f"{a} : {b} :: {c} : ?",
            "Top 1 entradas normalizadas": normalizada[0][0],
            "Top 1 vectores originales": original[0][0],
            "Mismo top 5": [w for w, _ in normalizada] == [w for w, _ in original],
        })
comparacion_operaciones = pd.DataFrame(filas_operacion)
display(comparacion_operaciones)

,Modelo,Consulta,Top 1 entradas normalizadas,Top 1 vectores originales,Mismo top 5
0,SVD-PPMI,casa : dinero :: trabajo : ?,apoyo,apoyo,False
1,SVD-PPMI,dinero : trabajo :: gobierno : ?,administración,administración,False
2,SVD-PPMI,trabajo : gobierno :: empresa : ?,firma,firma,False
3,Word2Vec noticias,casa : dinero :: trabajo : ?,garantías,vivir,False
4,Word2Vec noticias,dinero : trabajo :: gobierno : ?,congreso,congreso,False
5,Word2Vec noticias,trabajo : gobierno :: empresa : ?,autoridad,autoridad,False
6,Word2Vec Quijote,casa : dinero :: trabajo : ?,igualar,dolor,False
7,Word2Vec Quijote,dinero : trabajo :: gobierno : ?,agravio,agravio,False
8,Word2Vec Quijote,trabajo : gobierno :: empresa : ?,ínsula,ínsula,False
9,fastText SUC,casa : dinero :: trabajo : ?,remunerado,incentivo,False


### Cinco analogías sin excluir la consulta

Seleccioné cinco analogías priorizando la cantidad de modelos con cobertura de sus tres palabras de consulta y resolví los empates por el orden original. Este criterio consideró solo cobertura, no aciertos. Registré las consultas OOV y conservé exactamente las mismas cinco relaciones para todos.

Repetí la operación manteniendo las palabras de consulta entre los candidatos. Comparé el primer resultado con y sin exclusión y señalé si pertenecía a la consulta. Esto me permitió examinar por qué la aritmética vectorial es una aproximación y por qué una palabra de entrada puede dominar el resultado.

In [79]:
ids_consulta_comun = [
    i for i, fila in analogias_propias.iterrows()
    if all(fila[col] in vocabulario_comun_5 for col in ["a", "b", "c"])
]
cobertura_consultas = {
    i: sum(all(fila[col] in modelo.key_to_index for col in ["a", "b", "c"])
           for modelo in modelos_seccion5.values())
    for i, fila in analogias_propias.iterrows()
}
ids_sin_exclusion = sorted(
    analogias_propias.index, key=lambda i: (-cobertura_consultas[i], i)
)[:5]
print("Analogías seleccionadas:", ids_sin_exclusion)
filas_sin_exclusion = []
for id_analogia in ids_sin_exclusion:
    fila = analogias_propias.loc[id_analogia]
    a, b, c = (fila[col] for col in ["a", "b", "c"])
    for nombre, modelo in modelos_seccion5.items():
        faltantes = [p for p in (a, b, c) if p not in modelo.key_to_index]
        sin = analogia(a, b, c, nombre_modelo=nombre, excluir=False) if not faltantes else []
        con = analogia(a, b, c, nombre_modelo=nombre, excluir=True) if not faltantes else []
        filas_sin_exclusion.append({
            "Analogía": id_analogia, "Modelo": nombre,
            "Consulta": f"{a} : {b} :: {c} : ?", "Respuesta": fila["Respuesta"],
            "Primera sin excluir": sin[0][0] if sin else "Consulta OOV",
            "¿Primera es entrada?": sin[0][0] in {a, b, c} if sin else None,
            "Primera excluyendo": con[0][0] if con else "Consulta OOV",
            "Top 5 sin excluir": "; ".join(f"{w} ({s:.4f})" for w, s in sin),
            "Palabras OOV": ", ".join(faltantes),
        })
analogias_sin_exclusion = pd.DataFrame(filas_sin_exclusion)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(analogias_sin_exclusion)

Analogías seleccionadas: [4, 7, 10, 18, 1]


,Analogía,Modelo,Consulta,Respuesta,Primera sin excluir,¿Primera es entrada?,Primera excluyendo,Top 5 sin excluir,Palabras OOV
0,4,SVD-PPMI,españa : madrid :: francia : ?,parís,madrid,True,sede,madrid (0.6659); francia (0.6405); sede (0.4579); alemania (0.4269); debajo (0.4020),
1,4,Word2Vec noticias,españa : madrid :: francia : ?,parís,madrid,True,luxemburgo,madrid (0.7141); luxemburgo (0.6980); bélgica (0.6950); eslovenia (0.6925); francia (0.6894),
2,4,Word2Vec Quijote,españa : madrid :: francia : ?,parís,madrid,True,otubre,madrid (0.8536); otubre (0.7919); ocho (0.7851); millas (0.7841); treinta (0.7828),
3,4,fastText SUC,españa : madrid :: francia : ?,parís,toulouse,False,toulouse,toulouse (0.8946); parís (0.8866); lille (0.8524); montpellier (0.8369); rennes (0.8341),
4,7,SVD-PPMI,españa : español :: francia : ?,francés,español,True,europeo,español (0.6388); europeo (0.5081); francia (0.4844); industrial (0.4715); inmobiliario (0.4693),
5,7,Word2Vec noticias,españa : español :: francia : ?,francés,ibex,False,ibex,ibex (0.7021); stoxx (0.7007); terciario (0.6803); 312 (0.6690); 162 (0.6662),
6,7,Word2Vec Quijote,españa : español :: francia : ?,francés,español,True,amador,español (0.8244); amador (0.7857); sospira (0.7765); bartolomé (0.7723); forastero (0.7711),
7,7,fastText SUC,españa : español :: francia : ?,francés,francés,False,francés,francés (0.9235); español (0.7509); alemán (0.7425); français (0.7419); occitano (0.7168),
8,10,SVD-PPMI,empresa : empresas :: banco : ?,bancos,empresas,True,entidades,empresas (0.7169); entidades (0.6078); compañías (0.5434); personas (0.5405); banco (0.5322),
9,10,Word2Vec noticias,empresa : empresas :: banco : ?,bancos,empresas,True,central,empresas (0.6237); banco (0.5844); central (0.4869); entidades (0.4398); occidente (0.4346),


## 5.4 Análisis de mis resultados

### Verificación de mis funciones

Verifiqué mi función coseno en los pares `empresa–compañía`, `gobierno–estado` y `casa–dinero`. Las **12 comprobaciones**, tres por modelo, coincidieron con gensim dentro de la tolerancia numérica. Las diferencias mostradas fueron del orden de $10^{-8}$.

También comprobé mi función de analogías con tres consultas compartidas por los cuatro modelos. En las **12 verificaciones** obtuve el mismo top 5 y cosenos equivalentes a `most_similar`. Esto confirma la coincidencia de las implementaciones en esas consultas; no significa que sus resultados sean semánticamente correctos.

### Vecinos de los cuatro modelos

Comparé `banco`, `gobierno`, `empresa`, `casa` y `dinero`, presentes en los cuatro vocabularios. Encontré estas diferencias:

| Consulta | SVD-PPMI | Word2Vec noticias | Word2Vec Quijote | fastText SUC |
|---|---|---|---|---|
| banco | bbva, central, grupo | central, popular, república | encajó, ristre, correa | hipotecario, bancos, citibank |
| gobierno | congreso, país, federal | congreso, consejo, mandatario | condado, cobrar, ahorrar | ministro, gobiernos y grafías anómalas |
| empresa | compañía, plataforma, tecnología | startup, compañía, emergente | herida, demanda, novedad | compañía, constructora, subsidiaria |
| casa | coche, juntos, fuerza | pasiva, caminar, teletrabajo | aldea, corte, hacienda | mansión, casita, cabaña |
| dinero | información, manejo, préstamo | vivir, estorbo, crediticio | arráez, hallazgo, dueño | dinerito, dineral, centavo |

*En esta tabla resumí algunos vecinos para facilitar la comparación; las salidas anteriores conservan el top 5 completo, sus posiciones y cosenos.*

SVD y Word2Vec de noticias compartieron asociaciones del ámbito financiero y empresarial, pero no devolvieron listas idénticas. Para `banco`, reconocí relaciones bancarias en ambos; para `empresa`, ambos recuperaron `compañía`. El Quijote produjo contextos distintos y, en varias consultas modernas, listas poco coherentes. No interpreté sus cosenos altos como garantía de calidad: `banco` tuvo vecinos con cosenos superiores a 0.92 sin formar un grupo bancario reconocible.

SUC ofreció vecinos interpretables para `casa`, `empresa` y `dinero`. Sin embargo, también encontré `elgobierno`, `gobier` y una variante de `gobierno` con un carácter de separación interno. Por eso no consideré que un corpus grande eliminara automáticamente el ruido del vocabulario. Además, cada modelo buscó vecinos en un conjunto de candidatos de tamaño diferente.

### Antónimos: compartir contexto no significa ser sinónimos

| Par | SVD-PPMI | Word2Vec noticias | Word2Vec Quijote | fastText SUC |
|---|---:|---:|---:|---:|
| bueno / malo | OOV | 0.702509 | 0.799807 | 0.717445 |
| subir / bajar | OOV | 0.744858 | 0.766885 | 0.937153 |
| ganar / perder | OOV | 0.645275 | 0.656479 | 0.689466 |
| público / privado | 0.553847 | 0.657548 | OOV | 0.653345 |

Encontré similitudes positivas y considerables en los pares disponibles. Destacó `subir–bajar` en SUC, con **0.937153**. Entendí que estos vectores representan semejanzas de uso: palabras opuestas pueden aparecer con los mismos sujetos, objetos y temas. Por eso no interpreté similitud distribucional como sinonimia. Tampoco asigné un coseno de cero a los casos OOV, porque no pude calcularlos.

### Qué modelo se acercó más a mi juicio

Comparé los cosenos con los puntajes que definí para mis 20 pares:

| Modelo | Pares evaluados | Pares OOV | Spearman sobre su cobertura | Spearman sobre los 6 pares comunes |
|---|---:|---:|---:|---:|
| SVD-PPMI | 12 | 8 | 0.20215 | 0.35309 |
| Word2Vec noticias | 16 | 4 | 0.39705 | 0.55907 |
| Word2Vec Quijote | 13 | 7 | 0.36289 | 0.38252 |
| fastText SUC | 20 | 0 | 0.72641 | 0.26482 |

Con la cobertura propia de cada modelo, **SUC se acercó más a mi referencia**, con una correlación de **0.72641**, y fue el único que cubrió los 20 pares. Por ejemplo, distinguió `empresa–compañía` (**0.82390**) de `gobierno–zapato` (**0.07471**).

Al restringir la comparación a los **mismos seis pares**, obtuve la mayor correlación con **Word2Vec de noticias: 0.55907**. Esos pares fueron `empresa–compañía`, `dinero–moneda`, `gobierno–estado`, `banco–dinero`, `gobierno–ley` y `casa–familia`.

No interpreté este cambio como una contradicción: las dos columnas evalúan conjuntos diferentes. En el conjunto común desaparecieron todos los pares que puntué con 0 o 1 y quedaron principalmente relaciones cercanas o temáticas. Con solo seis observaciones, el orden entre modelos es sensible a pocos cambios. Por ello no concluí que Word2Vec superara a SUC de manera general, ni presenté las correlaciones sobre coberturas distintas como una comparación completamente controlada.

### Aciertos y cobertura de las analogías

Evalué **18 analogías de seis tipos**. Separé los aciertos entre los casos evaluables de los porcentajes sobre todas las propuestas:

| Modelo | Evaluables de 18 | Casos OOV | Top 1 entre evaluables (%) | Top 5 entre evaluables (%) | Top 1 sobre 18 (%) | Top 5 sobre 18 (%) |
|---|---:|---:|---:|---:|---:|---:|
| SVD-PPMI | 4 | 14 | 25.00 | 25.00 | 5.56 | 5.56 |
| Word2Vec noticias | 14 | 4 | 7.14 | 7.14 | 5.56 | 5.56 |
| Word2Vec Quijote | 9 | 9 | 11.11 | 11.11 | 5.56 | 5.56 |
| fastText SUC | 18 | 0 | 83.33 | 100.00 | 83.33 | 100.00 |

SUC recuperó **15 respuestas en la primera posición y las 18 dentro del top 5**. Cada modelo propio acertó una analogía en top 1: SVD recuperó `mercados` en `gobierno : gobiernos :: mercado : ?`; Word2Vec de noticias obtuvo `mexicano` en `colombia : colombiano :: méxico : ?`; el Quijote obtuvo `mala` en `bueno : buena :: malo : ?`.

Para comparar por tipo, resumí los porcentajes como **top 1 / top 5 sobre las tres propuestas de cada tipo**, contando los casos OOV como fallos de cobertura:

| Tipo | SVD-PPMI (%) | Word2Vec noticias (%) | Word2Vec Quijote (%) | SUC (%) |
|---|---:|---:|---:|---:|
| Género y parentesco | 0 / 0 | 0 / 0 | 0 / 0 | 100 / 100 |
| País-capital | 0 / 0 | 0 / 0 | 0 / 0 | 33.33 / 100 |
| País-gentilicio | 0 / 0 | 33.33 / 33.33 | 0 / 0 | 100 / 100 |
| Singular-plural | 33.33 / 33.33 | 0 / 0 | 0 / 0 | 100 / 100 |
| Tiempo verbal | 0 / 0 | 0 / 0 | 0 / 0 | 66.67 / 100 |
| Adjetivo masculino-femenino | 0 / 0 | 0 / 0 | 33.33 / 33.33 | 100 / 100 |

En SUC, las relaciones de parentesco, gentilicios, plurales y género de adjetivos funcionaron mejor en top 1. Las capitales fueron menos exactas: `parís` y `lima` quedaron en posición **2**, detrás de `toulouse` y `arequipa`. En tiempos verbales, `vivirá` también quedó en posición **2**, detrás de `vive`.

En los modelos propios distinguí la falta de cobertura de un error de ordenamiento. Ninguno pudo evaluar los tres ejemplos de tiempos verbales completos, así que sus ceros sobre las propuestas no prueban por sí solos que no aprendieran ninguna relación verbal. En cambio, Word2Vec de noticias sí tenía las palabras de `empresa : empresas :: banco : bancos`, pero situó `bancos` en posición **17**, fuera del top 5. Para `hombre : rey :: mujer : reina`, situó `reina` en posición **5,118**; el Quijote la colocó en **19** y SUC en **1**.

Solo hubo **dos analogías evaluables por los cuatro modelos**: `empresa : empresas :: banco : bancos` y `alto : alta :: bajo : baja`. SUC acertó ambas en top 1 y los modelos propios no las recuperaron en top 5. Este resultado favoreció a SUC sobre ese conjunto común, aunque dos casos son insuficientes para generalizar a todos los tipos de relación.

### El efecto de normalizar antes de combinar

Al comparar las doce consultas de verificación, encontré que usar vectores originales o normalizar cada entrada cambió el **top 5 en 11 de 12 casos** y la primera respuesta en **3 de 12**. Por ejemplo, en noticias, `casa : dinero :: trabajo : ?` pasó de `garantías` con entradas normalizadas a `vivir` con vectores originales.

Concluí que normalizar las palabras candidatas para calcular cosenos no equivale a normalizar las entradas antes de sumarlas. Dejé explícita la convención utilizada: para coincidir con gensim, evalué las analogías con entradas normalizadas. No interpreté las consultas de verificación como analogías semánticas correctas; su función fue comprobar la implementación.

### Qué ocurrió al no excluir las palabras de consulta

Repetí cinco relaciones, elegidas por cobertura de sus entradas, manteniendo las palabras de consulta entre los candidatos. Estos fueron los primeros resultados sin exclusión:

| Consulta | SVD-PPMI | Word2Vec noticias | Word2Vec Quijote | SUC |
|---|---|---|---|---|
| españa : madrid :: francia | madrid | madrid | madrid | toulouse |
| españa : español :: francia | español | ibex | español | francés |
| empresa : empresas :: banco | empresas | empresas | arcabuces | banco |
| alto : alta :: bajo | bajo | alta | alta | bajo |
| hombre : rey :: mujer | OOV | mujer | rey | reina |

Encontré que una palabra de entrada ocupó el primer lugar en **14 de las 19 consultas calculables**. Incluso SUC devolvió `banco` antes que `bancos` al permitir las entradas, con cosenos muy próximos: **0.7768** y **0.7766**. En `alto : alta :: bajo : ?`, SUC priorizó `bajo` y la respuesta esperada `baja` quedó tercera sin exclusión.

Esto me mostró que la operación conserva componentes fuertes de las palabras originales y puede favorecerlas. Excluirlas cambia el conjunto de candidatos, pero no garantiza recuperar la respuesta correcta. El signo **«≈» expresa una aproximación geométrica**, no una igualdad: en SUC obtuve `reina` como primera respuesta, pero en mis modelos Word2Vec esa misma relación falló. Mis conclusiones corresponden a estas consultas y a los vocabularios cargados, no a una evaluación exhaustiva del español.
